# HP Stock Harmonisation

Estimates the number of **heating heat pumps in operation** per Austrian
municipality in the reference year 2024 (reference date 2024-12-31).

Method: a two-stage, mass-preserving, proxy-based spatial disaggregation

```
national heating-heat-pump stock (Austria)
    -> federal state, weighted by reported heat-pump households (2023/2024)
    -> municipality, weighted by residential buildings
```

System boundary: **heating** heat pumps only. Domestic-hot-water-only heat
pumps, ventilation heat pumps and industrial heat pumps are reported as
separate categories in the national source and are **excluded** here.

Out of scope for this workflow (later scenario stages of the thesis):
electrical rating, thermal rating, COP, seasonal performance factor,
temperature profiles, temporal load profiles, building-level selection,
grid-connection-point allocation, and future scenarios. None of these is
implemented, and none of them may be inferred from the outputs of this
notebook.

The national stock is **not** parsed from the market-report PDF. It is
entered manually in the configuration cell of Section 3 and is the single
binding national control total of this workflow.

All municipality and federal-state values produced here are **modelled
estimates**, not observed heat-pump counts.

## 1. Install packages

In [1]:
%pip install pandas openpyxl odfpy pyarrow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\friha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Imports and paths

In [2]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display


CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find the project data folder. "
        "Run the notebook from the project root or the notebooks folder."
    )

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Data folder:", DATA_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())

Project root: C:\Users\friha\TU Wien\Diplomarbeit\00_Harmonisierung\00_Harmonisierung_HP
Data folder: C:\Users\friha\TU Wien\Diplomarbeit\00_Harmonisierung\00_Harmonisierung_HP\data
Output folder: C:\Users\friha\TU Wien\Diplomarbeit\00_Harmonisierung\00_Harmonisierung_HP\outputs


## 3. Configuration

### National heating-heat-pump stock — source of the manually entered value

`NATIONAL_HEATING_HP_STOCK` is transcribed by hand from the Austrian market
report. The notebook does **not** read, parse or otherwise depend on the PDF;
the PDF is a bibliographic reference only.

| Item | Value |
|---|---|
| Source | *Innovative Energietechnologien in Österreich – Marktentwicklung 2024. Biomasse, Photovoltaik, Solarthermie, Wärmepumpen und Windkraft* |
| Series | Berichte aus Energie- und Umweltforschung 23/2025 |
| Publisher (Medieninhaber, Verleger und Herausgeber) | Bundesministerium für Innovation, Mobilität und Infrastruktur (BMIMI), Radetzkystraße 2, 1030 Wien |
| Scientific project lead / editor, report part heat pumps | ENFOS e.U., Peter Biermayr |
| Place, date | Vienna, May 2025 |
| Report part | Chapter 9, *Wärmepumpen* |
| Section on installations in operation | 9.1.3 *In Betrieb befindliche Anlagen* (report page 147) |
| Table on the development of the heat-pump stock | Table 51, *Entwicklung des Wärmepumpenbestandes in Österreich* (report page 150) |
| Row used | year **2024** |
| Category used | column **Heizung** (heating heat pumps) |
| Value used | **407,637** installations |
| File in this project | `data/BMIMI-marktstatistik-2024.pdf` (cited, not parsed) |

The same value is also stated in the running text of Section 9.1.3 and in the
summary line of Table 50 (*Entwicklung des Wärmepumpen-Inlandsmarktes in
Österreich*, "Annahme 20 Jahre Lebensdauer: Betrachtungszeitraum 2005–2024").

**The national stock is itself a modelled stock.** The report derives it from
historical annual domestic sales figures and an assumed technical service life
of 20 years; installations older than the assumed service life are treated as
no longer in operation. The report explicitly states that no Weibull-distributed
retirement function was implemented, because the data required to parameterise
it empirically were not available. The value is therefore a transparent
model result, not an observed census of installed units.

Categories reported separately for 2024 in the same table and **excluded**
here: domestic hot water (*Brauchwasser*) 121,616; ventilation
(*WRL & Luft/Luft*) 6,775; industrial (*Industrie*) 935. Total across all
categories: 536,963.

In [3]:
# ---------------------------------------------------------------------------
# MANUAL NATIONAL CONTROL TOTAL
# ---------------------------------------------------------------------------
# Transcribed by hand from the market report documented in the markdown cell
# above (Table 51, row 2024, column "Heizung"). This is the single binding
# national stock total of this workflow. It is NOT extracted from the PDF
# programmatically, and no other national total may override it.
NATIONAL_HEATING_HP_STOCK = 407_637

REFERENCE_YEAR = 2024
REFERENCE_DATE = "2024-12-31"

# pandas timestamp used for the reference_date column, matching the dtype
# convention of the sibling PV and EV harmonisation outputs.
REFERENCE_DATE_TS = pd.Timestamp(REFERENCE_DATE)

# The federal-state proxy (heat-pump households) comes from the Statistik
# Austria microcensus survey period 2023/2024. This is deliberately tracked
# as its own constant: it is NOT identical to the reference date of the
# national stock and must not be equated with it.
STATE_PROXY_SURVEY_PERIOD = "2023/2024"

# The municipal building proxy comes from the Abgestimmte Erwerbsstatistik
# (register-based labour market and housing statistics) for 2024, territorial
# status 2026 - the same territorial status as the canonical municipality list.
BUILDING_PROXY_YEAR = 2024
BUILDING_PROXY_TERRITORIAL_STATUS = 2026

# ---------------------------------------------------------------------------
# Input paths
# ---------------------------------------------------------------------------
HP_HOUSEHOLDS_PATH = DATA_DIR / "GesamteinsatzAllerEnergietraeger2003Bis2024.ods"
HP_ENERGY_VALIDATION_PATH = DATA_DIR / "Energieeinsatz_HP_Bundesländer.xlsx"
BUILDINGS_PATH = DATA_DIR / "AEST_Gemeindeergebnisse_2024.ods"
MUNICIPALITIES_PATH = DATA_DIR / "municipalities_clean.xlsx"

# Cited as the source of NATIONAL_HEATING_HP_STOCK, never opened by this
# notebook. Its absence must not break the workflow.
MARKET_REPORT_PATH_FOR_CITATION_ONLY = DATA_DIR / "BMIMI-marktstatistik-2024.pdf"

# ---------------------------------------------------------------------------
# Output paths
# ---------------------------------------------------------------------------
OUTPUT_XLSX_PATH = OUTPUT_DIR / "hp_municipality_harmonised.xlsx"
AUDIT_PARQUET_PATH = OUTPUT_DIR / "hp_municipality_audit.parquet"

# ---------------------------------------------------------------------------
# Controlled vocabulary for the canonical output schema
# ---------------------------------------------------------------------------
DERIVATION_METHOD = (
    "national_heating_hp_stock_state_hp_households_municipal_residential_buildings"
)
DATA_QUALITY = "modelled_mass_preserving"

# ---------------------------------------------------------------------------
# Federal states
# ---------------------------------------------------------------------------
# Key: first digit of the Gemeindekennziffer (GKZ) = federal-state code.
# Value: federal-state name, which is also the sheet name in the household
# source workbook and the column label in the energy validation workbook.
# The name is used as `federal_state_code` in the canonical output, following
# the sibling EV harmonisation workflow.
GKZ_PREFIX_TO_STATE_NAME = {
    "1": "Burgenland",
    "2": "Kärnten",
    "3": "Niederösterreich",
    "4": "Oberösterreich",
    "5": "Salzburg",
    "6": "Steiermark",
    "7": "Tirol",
    "8": "Vorarlberg",
    "9": "Wien",
}
NATIONAL_SHEET_NAME = "Österreich"
HEAT_PUMP_ROW_LABEL = "Wärmepumpe"

REQUIRED_INPUT_PATHS = {
    "GesamteinsatzAllerEnergietraeger2003Bis2024.ods (federal-state heat-pump households)": HP_HOUSEHOLDS_PATH,
    "Energieeinsatz_HP_Bundesländer.xlsx (validation-only energy cross-check)": HP_ENERGY_VALIDATION_PATH,
    "AEST_Gemeindeergebnisse_2024.ods (municipal building stock)": BUILDINGS_PATH,
    "municipalities_clean.xlsx (canonical municipality master data)": MUNICIPALITIES_PATH,
}
missing_required_inputs = [
    f"{label}: {path.resolve()}"
    for label, path in REQUIRED_INPUT_PATHS.items()
    if not path.exists()
]
if missing_required_inputs:
    raise FileNotFoundError(
        "The following required input file(s) are missing:\n"
        + "\n".join(f"  - {entry}" for entry in missing_required_inputs)
    )

assert NATIONAL_HEATING_HP_STOCK == 407_637, (
    "NATIONAL_HEATING_HP_STOCK was changed from the documented source value."
)

print("All required input files were found.")
print("NATIONAL_HEATING_HP_STOCK:", f"{NATIONAL_HEATING_HP_STOCK:,}")
print("REFERENCE_YEAR:", REFERENCE_YEAR)
print("REFERENCE_DATE:", REFERENCE_DATE)
print("STATE_PROXY_SURVEY_PERIOD:", STATE_PROXY_SURVEY_PERIOD)
print("BUILDING_PROXY_YEAR:", BUILDING_PROXY_YEAR,
      "| territorial status:", BUILDING_PROXY_TERRITORIAL_STATUS)
print()
print("Market report (cited only, never parsed):",
      MARKET_REPORT_PATH_FOR_CITATION_ONLY.name,
      "- present:", MARKET_REPORT_PATH_FOR_CITATION_ONLY.exists())

All required input files were found.
NATIONAL_HEATING_HP_STOCK: 407,637
REFERENCE_YEAR: 2024
REFERENCE_DATE: 2024-12-31
STATE_PROXY_SURVEY_PERIOD: 2023/2024
BUILDING_PROXY_YEAR: 2024 | territorial status: 2026

Market report (cited only, never parsed): BMIMI-marktstatistik-2024.pdf - present: True


## 4. Method, notation and literature

### 4.1 Two-stage mass-preserving disaggregation

$$
\text{Austria} \;\longrightarrow\; \text{federal states} \;\longrightarrow\; \text{municipalities}
$$

Stage 1 splits the national stock across federal states in proportion to the
reported number of households using a heat pump. Stage 2 splits each
federal-state stock across its municipalities in proportion to the number of
residential buildings. Both stages preserve mass exactly: no installation is
created or lost, and integer allocation is performed with the Largest
Remainder method.

### 4.2 Notation

Every symbol is also restated directly next to the formula where it is first
used, so that no formula depends on this table alone.

| Symbol | Meaning |
|---|---|
| $N_{\mathrm{AT}}^{HP}$ | national stock of heating heat pumps in Austria |
| $\mathcal{S}$ | set of all Austrian federal states |
| $s, r$ | a federal state; $r$ is the running index over $\mathcal{S}$ |
| $H_s^{HP}$ | number of households using a heat pump in federal state $s$ |
| $w_s^{HP}$ | share of federal state $s$ in all reported heat-pump households |
| $\widetilde{N}_s^{HP}$ | unrounded modelled heat-pump stock of federal state $s$ |
| $N_s^{HP}$ | final integer heat-pump stock of federal state $s$ |
| $E_s^{HP}$ | total heat-pump energy input of federal state $s$ (GJ) |
| $\bar{e}_s^{HP}$ | average energy input per heat-pump household in $s$ (GJ) |
| $\mathcal{M}_s$ | set of all municipalities in federal state $s$ |
| $m, j$ | a municipality; $j$ is the running index over $\mathcal{M}_s$ |
| $B_m^{1}, B_m^{2}, B_m^{3+}$ | residential buildings in $m$ with one, two, and three or more dwellings |
| $B_m^{res}$ | total number of residential buildings in municipality $m$ |
| $X_m^{HP}$ | municipal proxy value for the heat-pump distribution |
| $q_{m\mid s}^{HP}$ | share of municipality $m$ in the total proxy value of its federal state $s$ |
| $\widetilde{N}_m^{HP}$ | unrounded modelled heat-pump stock of municipality $m$ |
| $N_m^{HP}$ | final integer heat-pump stock of municipality $m$ |

### 4.3 Methodological literature

The four publications below are located in the shared
`00_Harmonisierung/literature/` directory, outside this project folder. They
are cited bibliographically; the notebook does not read or parse them.

- **Patil et al. (2024)**, *A systematic review of spatial disaggregation
  methods for climate action planning*, supports classifying the procedure
  implemented here as a **proxy-data-based, mass-preserving spatial
  disaggregation** — one of the principal method families identified in that
  review, alongside machine-learning and geostatistical model-based
  approaches.
- **Comber & Zeng (2019)**, *Spatial interpolation using areal features*,
  supports the use of **suitable ancillary information** instead of naive
  areal (area-proportional) weighting. Here the ancillary information is the
  municipal residential building stock, not municipal area.
- **Marinello et al. (2024)**, *Downscaling atmospheric emission inventories
  with "top–down" approach*, supports the requirement that a top-down proxy
  variable must be **substantively related to the target quantity**. A heating
  heat pump is installed in a building; the residential building stock is
  therefore the substantively appropriate proxy family for this target
  quantity.
- **Zapparoli et al. (2025)**, *Future Deployment and Flexibility of
  Distributed Energy Resources in the Distribution Grids of Switzerland*,
  allocates heat pumps to individual buildings taken from a building
  registry and derives their rating from building attributes
  (construction period, construction type, heating area). This supports the
  use of **building-register information as the carrier of heat-pump
  allocation**, and it locates rating, COP and load-profile modelling in the
  later scenario stage — which is explicitly outside the scope of this
  notebook.

These publications support the **method category** and the choice of a
building-related proxy. **They do not prescribe the specific Austrian
distribution key used here.** The concrete calculation is an own, transparently
documented application to the data available in this project.

## 5. General helper functions

In [4]:
def parse_numeric(series: pd.Series) -> pd.Series:
    """Parse Statistik Austria numeric cells to float.

    Removes non-breaking spaces, narrow no-break spaces and ordinary spaces
    used as thousands separators. Deliberately does NOT translate commas to
    decimal points: the source files export decimal points, so a comma would
    indicate an unexpected locale and must surface as a parse failure rather
    than be silently reinterpreted. Values that cannot be parsed become NaN
    and are caught by the explicit checks at each call site.
    """
    cleaned = (
        series.astype(str)
        .str.replace(" ", "", regex=False)  # non-breaking space
        .str.replace(" ", "", regex=False)  # narrow no-break space
        .str.replace(" ", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")


def locate_unique_column(header_row: pd.Series, label: str) -> int:
    """Return the single column index whose stripped header equals `label`.

    Raises if the label is absent or ambiguous, so that a changed source
    layout stops the workflow instead of silently shifting a column.
    """
    matches = [
        position
        for position, value in enumerate(header_row.tolist())
        if str(value).strip() == label
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one column labelled '{label}', found {len(matches)}."
        )
    return matches[0]


def largest_remainder_apportionment(
    values: pd.Series,
    total: int,
    secondary_key: pd.Series | None = None,
) -> pd.Series:
    """Integer, sum-preserving allocation of `total` using the Largest
    Remainder method.

    `values` holds the unrounded real-valued shares of `total` and is indexed
    by the unit identifier (federal-state code in Stage 1, municipality_id in
    Stage 2).

    Procedure: floor every value, compute the decimal remainders, compute the
    number of still-missing units, and award those units to the entries with
    the largest remainders.

    Tie-breaking is an explicit compound sort key and does not rely on the
    input order of `values` or on sort stability:

    1. fractional remainder, descending;
    2. `secondary_key`, descending (the municipal proxy value in Stage 2;
       constant, and therefore inert, when no secondary key is supplied);
    3. unit identifier, ascending.

    The result is therefore identical regardless of the order in which units
    appear in `values`. No random allocation is used anywhere.
    """
    if not values.index.is_unique:
        raise ValueError("largest_remainder_apportionment requires a unique index.")
    if (values < 0).any():
        raise ValueError("largest_remainder_apportionment requires non-negative values.")

    floor_values = np.floor(values).astype("int64")
    remainder = values - floor_values

    shortfall = int(total) - int(floor_values.sum())
    if shortfall < 0:
        raise AssertionError(
            "Largest Remainder allocation failed: the sum of floor values "
            "exceeds the target total."
        )
    if shortfall > len(values):
        raise AssertionError(
            "Largest Remainder allocation failed: more units are missing than "
            "there are entries to distribute them over."
        )

    if secondary_key is None:
        secondary_values = np.zeros(len(values), dtype=float)
    else:
        secondary_values = secondary_key.reindex(values.index).to_numpy(dtype=float)
        if np.isnan(secondary_values).any():
            raise ValueError("secondary_key does not cover every entry of `values`.")

    # The source index is deliberately not reused as the DataFrame index:
    # keeping the identifier as a plain column avoids any ambiguity between an
    # index level and a column of the same name when sorting.
    tie_break_key = pd.DataFrame(
        {
            "remainder": remainder.to_numpy(),
            "secondary": secondary_values,
            "unit_id": values.index.to_numpy(),
        }
    )
    ranked_positions = tie_break_key.sort_values(
        ["remainder", "secondary", "unit_id"],
        ascending=[False, False, True],
        kind="mergesort",
    ).index
    winners = values.index[ranked_positions][:shortfall]

    result = floor_values.copy()
    result.loc[winners] += 1

    assert int(result.sum()) == int(total)
    return result

### 5a. Deterministic tie-break test

Two small artificial tests verify that the Largest Remainder tie-break is
determined solely by the explicit compound key — (fractional remainder
descending, secondary key descending, identifier ascending) — and not by the
order in which entries happen to appear in the input `Series`. Each test is
run in two different input orderings and both must produce the identical
result.

Test 1 reproduces the Stage 1 situation (no secondary key): among equal
remainders the smallest identifier wins. Test 2 reproduces the Stage 2
situation: among equal remainders the larger proxy value wins, and only where
both remainder and proxy are equal does the smallest identifier decide.

In [5]:
def _run_tie_break_tests():
    # Test 1 - Stage 1 style, no secondary key, three tied values.
    unit_ids = ["3", "1", "2"]  # deliberately not sorted
    for order in (unit_ids, list(reversed(unit_ids))):
        test_values = pd.Series([2.5] * 3, index=order)
        result = largest_remainder_apportionment(test_values, total=8)
        # floors = [2, 2, 2] (sum 6), shortfall 2 -> the two smallest ids win.
        expected = pd.Series({"1": 3, "2": 3, "3": 2})
        actual = result.reindex(expected.index)
        assert actual.equals(expected), (
            f"Stage 1 tie-break test failed for input order {order}: "
            f"expected {expected.to_dict()}, got {actual.to_dict()}"
        )
        assert int(actual.sum()) == 8

    # Test 2 - Stage 2 style, equal remainders, secondary proxy key decides;
    # two entries share both remainder and proxy so the identifier decides.
    municipality_ids = ["30003", "10001", "20002", "40004"]
    proxies = {"30003": 500.0, "10001": 100.0, "20002": 100.0, "40004": 900.0}
    for order in (municipality_ids, list(reversed(municipality_ids))):
        test_values = pd.Series([2.5] * 4, index=order)
        secondary = pd.Series(proxies).reindex(order)
        result = largest_remainder_apportionment(
            test_values, total=10, secondary_key=secondary
        )
        # floors = [2, 2, 2, 2] (sum 8), shortfall 2 -> largest proxies win:
        # 40004 (900) and 30003 (500).
        expected = pd.Series({"10001": 2, "20002": 2, "30003": 3, "40004": 3})
        actual = result.reindex(expected.index)
        assert actual.equals(expected), (
            f"Stage 2 tie-break test failed for input order {order}: "
            f"expected {expected.to_dict()}, got {actual.to_dict()}"
        )
        assert int(actual.sum()) == 10

    # Test 3 - equal remainder AND equal proxy: smallest identifier wins.
    tied_ids = ["20002", "10001"]
    for order in (tied_ids, list(reversed(tied_ids))):
        test_values = pd.Series([1.5] * 2, index=order)
        secondary = pd.Series({"10001": 7.0, "20002": 7.0}).reindex(order)
        result = largest_remainder_apportionment(
            test_values, total=3, secondary_key=secondary
        )
        expected = pd.Series({"10001": 2, "20002": 1})
        actual = result.reindex(expected.index)
        assert actual.equals(expected), (
            f"Identifier tie-break test failed for input order {order}: "
            f"expected {expected.to_dict()}, got {actual.to_dict()}"
        )

    print("Deterministic tie-break tests passed:")
    print("  1. equal remainders  -> smallest identifier wins")
    print("  2. equal remainders  -> larger proxy value wins")
    print("  3. equal remainders and equal proxy -> smallest identifier wins")
    print("  results are independent of input order in all three cases")


_run_tie_break_tests()

Deterministic tie-break tests passed:
  1. equal remainders  -> smallest identifier wins
  2. equal remainders  -> larger proxy value wins
  3. equal remainders and equal proxy -> smallest identifier wins
  results are independent of input order in all three cases


## 6. Load the canonical municipality master table

`data/municipalities_clean.xlsx` is an **unchanged byte-identical copy** of
the canonical municipality list of the sibling EV harmonisation workflow:

```text
original path: harmonisation_EV/data/municipalities_clean.xlsx
original name: municipalities_clean.xlsx
```

The copy exists so that the HP workflow is reproducible on its own. The EV
file is never read, written or modified by this notebook. The same
Gemeindekennziffern (GKZ), municipality names, federal-state codes and
territorial status are used. The GKZ is the primary and only join key of this
workflow — no name matching, no fuzzy matching, and no silent first-match
selection is used anywhere.

In [6]:
municipalities = pd.read_excel(MUNICIPALITIES_PATH, sheet_name="municipalities")

municipalities["municipality_id"] = (
    municipalities["municipality_id"].astype(str).str.zfill(5)
)
municipalities["federal_state_id"] = municipalities["municipality_id"].str[0]
municipalities["federal_state_code"] = municipalities["federal_state_id"].map(
    GKZ_PREFIX_TO_STATE_NAME
)
municipalities["reference_date"] = REFERENCE_DATE_TS

municipalities = municipalities[
    ["municipality_id", "municipality_name", "federal_state_id",
     "federal_state_code", "reference_date"]
].sort_values("municipality_id").reset_index(drop=True)

check_municipality_id_unique = bool(municipalities["municipality_id"].is_unique)
check_municipality_count = len(municipalities) == 2092
check_state_assignment_complete = bool(
    municipalities["federal_state_code"].notna().all()
)
check_municipality_names_present = bool(
    municipalities["municipality_name"].notna().all()
)
check_vienna_present_once = int(
    (municipalities["municipality_id"] == "90001").sum()
) == 1

assert check_municipality_id_unique, "municipality_id is not unique."
assert check_state_assignment_complete, "Unmapped federal-state GKZ prefix found."
assert check_municipality_names_present, "Missing municipality name found."

CANONICAL_MUNICIPALITY_IDS = set(municipalities["municipality_id"])

print("Municipalities in the canonical master:", len(municipalities))
print("municipality_id unique:", check_municipality_id_unique)
print("Municipality count equals 2092:", check_municipality_count)
print("Vienna (90001) present exactly once:", check_vienna_present_once)
print()
print("Municipalities per federal state:")
display(
    municipalities.groupby(["federal_state_id", "federal_state_code"])
    .size()
    .rename("municipalities")
    .reset_index()
)

Municipalities in the canonical master: 2092
municipality_id unique: True
Municipality count equals 2092: True
Vienna (90001) present exactly once: True

Municipalities per federal state:


,federal_state_id,federal_state_code,municipalities
0,1,Burgenland,171
1,2,Kärnten,132
2,3,Niederösterreich,573
3,4,Oberösterreich,438
4,5,Salzburg,119
5,6,Steiermark,285
6,7,Tirol,277
7,8,Vorarlberg,96
8,9,Wien,1


## 7. Load federal-state heat-pump households (Stage 1 proxy)

Source: `data/GesamteinsatzAllerEnergietraeger2003Bis2024.ods` (Statistik
Austria, *Gesamteinsatz aller Energieträger*, microcensus *Energieeinsatz der
Haushalte*). One sheet per federal state plus a national sheet; each sheet
contains one block per survey period. The block for **2023/2024** is used.

From the row `Wärmepumpe` the notebook reads the columns:

| Column in the source | Read as | Role |
|---|---|---|
| `Haushalte (HH)` | `hp_households` | $H_s^{HP}$ — **the reported number of households, used directly as the Stage 1 proxy** |
| `GJ` | `hp_energy_input_gj` | $E_s^{HP}$ — validation only (Section 9) |
| `GJ pro HH` | `gj_per_hp_household` | $\bar{e}_s^{HP}$ — validation only (Section 9) |

The household figure is taken as reported. It is **not** derived by dividing a
federal-state energy input by a nationwide average GJ per household; the GJ and
GJ-per-household columns never enter the allocation and are used exclusively
for the plausibility checks in Section 9.

Note on the source definition: households are counted once for **every** energy
carrier they use, so the carrier rows overlap by construction. `hp_households`
is the number of households that report using a heat pump — it is not a count
of heat-pump installations, and a household is not the same thing as an
installation. This limitation is carried through to the results.

In [7]:
def read_energy_carrier_block(sheet_name: str) -> dict:
    """Read the 2023/2024 heat-pump row from one federal-state sheet.

    The block, its header row and the required columns are located by their
    labels, not by hard-coded row or column offsets, so that a changed source
    layout raises instead of silently reading the wrong cells.
    """
    frame = pd.read_excel(
        HP_HOUSEHOLDS_PATH, sheet_name=sheet_name, engine="odf", header=None
    )

    title_rows = [
        row
        for row in range(len(frame))
        if str(frame.iloc[row, 0]).strip().startswith("Gesamteinsatz aller Energieträger")
    ]
    period_rows = [
        row for row in title_rows
        if STATE_PROXY_SURVEY_PERIOD in str(frame.iloc[row, 0])
    ]
    if len(period_rows) != 1:
        raise ValueError(
            f"Sheet '{sheet_name}': expected exactly one block for survey period "
            f"{STATE_PROXY_SURVEY_PERIOD}, found {len(period_rows)}."
        )
    title_row = period_rows[0]
    title_text = str(frame.iloc[title_row, 0]).strip()

    # Guard against a mislabelled or reordered sheet.
    if sheet_name not in title_text:
        raise ValueError(
            f"Sheet '{sheet_name}': block title does not name this territory "
            f"({title_text!r})."
        )

    later_titles = [row for row in title_rows if row > title_row]
    block_end = min(later_titles) if later_titles else len(frame)

    header_candidates = [
        row for row in range(title_row + 1, min(title_row + 6, block_end))
        if "Haushalte (HH)" in [str(v).strip() for v in frame.iloc[row].tolist()]
    ]
    if len(header_candidates) != 1:
        raise ValueError(
            f"Sheet '{sheet_name}': expected exactly one header row containing "
            f"'Haushalte (HH)', found {len(header_candidates)}."
        )
    header_row = header_candidates[0]
    header = frame.iloc[header_row]

    columns = {
        "hp_households": locate_unique_column(header, "Haushalte (HH)"),
        "hp_energy_input_gj": locate_unique_column(header, "GJ"),
        "gj_per_hp_household": locate_unique_column(header, "GJ pro HH"),
    }

    carrier_rows = [
        row for row in range(header_row + 1, block_end)
        if str(frame.iloc[row, 0]).strip() == HEAT_PUMP_ROW_LABEL
    ]
    if len(carrier_rows) != 1:
        raise ValueError(
            f"Sheet '{sheet_name}': expected exactly one '{HEAT_PUMP_ROW_LABEL}' "
            f"row in the {STATE_PROXY_SURVEY_PERIOD} block, found {len(carrier_rows)}."
        )
    carrier_row = carrier_rows[0]

    record = {
        "source_sheet": sheet_name,
        "survey_period": STATE_PROXY_SURVEY_PERIOD,
        "source_block_title": title_text,
    }
    for name, column in columns.items():
        value = parse_numeric(pd.Series([frame.iloc[carrier_row, column]])).iloc[0]
        if pd.isna(value):
            raise ValueError(
                f"Sheet '{sheet_name}': column '{name}' of the "
                f"'{HEAT_PUMP_ROW_LABEL}' row could not be parsed as a number."
            )
        record[name] = float(value)
    return record


state_records = []
for state_id, state_name in GKZ_PREFIX_TO_STATE_NAME.items():
    record = read_energy_carrier_block(state_name)
    record["federal_state_id"] = state_id
    record["federal_state_code"] = state_name
    state_records.append(record)

hp_households = (
    pd.DataFrame(state_records)
    .sort_values("federal_state_id")
    .reset_index(drop=True)
)

national_record = read_energy_carrier_block(NATIONAL_SHEET_NAME)

check_all_nine_states_present = len(hp_households) == 9
check_no_duplicate_state_codes = bool(
    hp_households["federal_state_id"].is_unique
    and hp_households["federal_state_code"].is_unique
)
check_no_negative_households = bool((hp_households["hp_households"] >= 0).all())

assert check_all_nine_states_present, "Expected exactly nine federal states."
assert check_no_duplicate_state_codes, "Duplicate federal-state code found."
assert check_no_negative_households, "Negative heat-pump household count found."

# The nine federal-state figures must reconstruct the national sheet of the
# same source. This validates the block/row/column location logic above.
state_household_sum = float(hp_households["hp_households"].sum())
national_household_value = float(national_record["hp_households"])
household_sum_deviation = state_household_sum - national_household_value
check_state_sum_matches_national_sheet = bool(
    abs(household_sum_deviation) <= 1e-6 * max(1.0, abs(national_household_value))
)

print(f"Survey period: {STATE_PROXY_SURVEY_PERIOD}")
print(f"Sum of the nine federal states : {state_household_sum:,.3f} households")
print(f"National sheet '{NATIONAL_SHEET_NAME}'   : {national_household_value:,.3f} households")
print(f"Deviation                      : {household_sum_deviation:,.9f}")
print("State sum reconstructs the national sheet:",
      check_state_sum_matches_national_sheet)
print()
display(
    hp_households[
        ["federal_state_id", "federal_state_code", "hp_households",
         "hp_energy_input_gj", "gj_per_hp_household"]
    ]
)

Survey period: 2023/2024
Sum of the nine federal states : 617,860.782 households
National sheet 'Österreich'   : 617,860.782 households
Deviation                      : -0.000000001
State sum reconstructs the national sheet: True



,federal_state_id,federal_state_code,hp_households,hp_energy_input_gj,gj_per_hp_household
0,1,Burgenland,39639.146560,8.810170e+05,22.225934
1,2,Kärnten,41888.742771,1.092810e+06,26.088400
2,3,Niederösterreich,173941.830919,4.321572e+06,24.844929
3,4,Oberösterreich,140525.839659,4.172924e+06,29.695066
4,5,Salzburg,32716.060608,9.015926e+05,27.558103
5,6,Steiermark,76566.211533,1.869720e+06,24.419644
6,7,Tirol,48914.492547,1.201818e+06,24.569765
7,8,Vorarlberg,38895.401949,8.957361e+05,23.029356
8,9,Wien,24773.055795,5.125615e+05,20.690283


## 8. Stage 1 — Austria to federal states

**Share of each federal state in all reported heat-pump households**

$$
w_s^{HP} \;=\; \frac{H_s^{HP}}{\sum_{r\in\mathcal{S}} H_r^{HP}}
$$

where $H_s^{HP}$ is the number of households using a heat pump in federal
state $s$, $r$ is the running index over all federal states, $\mathcal{S}$ is
the set of all nine Austrian federal states, and $w_s^{HP}$ is the share of
federal state $s$ in all reported heat-pump households.

**Unrounded modelled federal-state stock**

$$
\widetilde{N}_s^{HP} \;=\; N_{\mathrm{AT}}^{HP}\cdot w_s^{HP}
$$

where $N_{\mathrm{AT}}^{HP}$ is the manually configured national stock of
heating heat pumps (Section 3), $w_s^{HP}$ is the federal-state share from
above, and $\widetilde{N}_s^{HP}$ is the not-yet-rounded federal-state stock.

**Integer allocation with the Largest Remainder method**

1. round every unrounded value down;
2. determine the decimal remainders;
3. compute the number of installations still missing;
4. award the missing installations to the federal states with the largest
   remainders;
5. where remainders are equal, use the federal-state code in ascending order
   as a deterministic tie-break.

After rounding the following holds exactly:

$$
\sum_{s\in\mathcal{S}} N_s^{HP} \;=\; N_{\mathrm{AT}}^{HP} \;=\; 407\,637
$$

where $N_s^{HP}$ is the final integer heat-pump stock of federal state $s$.

> The resulting federal-state figures are **modelled estimates**, not observed
> stock counts. They combine a modelled national total with a survey-based
> household distribution; neither component is an installation census.

In [8]:
total_hp_households = float(hp_households["hp_households"].sum())
if total_hp_households <= 0:
    raise ValueError("Total heat-pump households is zero or negative.")

hp_households["hp_household_share"] = (
    hp_households["hp_households"] / total_hp_households
)
hp_households["hp_stock_raw"] = (
    NATIONAL_HEATING_HP_STOCK * hp_households["hp_household_share"]
)

state_raw = pd.Series(
    hp_households["hp_stock_raw"].to_numpy(),
    index=hp_households["federal_state_id"].to_numpy(),
)
state_final = largest_remainder_apportionment(state_raw, NATIONAL_HEATING_HP_STOCK)

hp_households["hp_stock_floor"] = np.floor(hp_households["hp_stock_raw"]).astype("int64")
hp_households["fractional_remainder"] = (
    hp_households["hp_stock_raw"] - hp_households["hp_stock_floor"]
)
hp_households["hp_stock_final"] = (
    hp_households["federal_state_id"].map(state_final).astype("int64")
)
hp_households["rounding_adjustment"] = (
    hp_households["hp_stock_final"] - hp_households["hp_stock_floor"]
).astype(bool)

federal_state_audit = hp_households[
    [
        "federal_state_id",
        "federal_state_code",
        "hp_households",
        "hp_household_share",
        "hp_energy_input_gj",
        "gj_per_hp_household",
        "hp_stock_raw",
        "hp_stock_floor",
        "fractional_remainder",
        "rounding_adjustment",
        "hp_stock_final",
    ]
].copy()
federal_state_audit["survey_period"] = STATE_PROXY_SURVEY_PERIOD
federal_state_audit["reference_date"] = REFERENCE_DATE_TS
federal_state_audit["value_status"] = "modelled_estimate_not_observed_stock"

STATE_STOCK = dict(
    zip(federal_state_audit["federal_state_id"], federal_state_audit["hp_stock_final"])
)

check_state_shares_sum_to_one = bool(
    abs(float(hp_households["hp_household_share"].sum()) - 1.0) <= 1e-12
)
check_state_stock_sums_to_national = (
    int(federal_state_audit["hp_stock_final"].sum()) == NATIONAL_HEATING_HP_STOCK
)

assert check_state_shares_sum_to_one, "Federal-state shares do not sum to 1."
assert check_state_stock_sums_to_national, (
    "Federal-state stocks do not sum to the national stock."
)

print(f"National heating heat-pump stock: {NATIONAL_HEATING_HP_STOCK:,}")
print(f"Sum of federal-state shares     : {hp_households['hp_household_share'].sum():.15f}")
print(f"Sum of final federal-state stock: {int(federal_state_audit['hp_stock_final'].sum()):,}")
print(f"Rounding adjustments (+1)       : {int(federal_state_audit['rounding_adjustment'].sum())}")
print()
display(
    federal_state_audit[
        ["federal_state_id", "federal_state_code", "hp_households",
         "hp_household_share", "hp_stock_raw", "hp_stock_floor",
         "fractional_remainder", "rounding_adjustment", "hp_stock_final"]
    ].style.format(
        {
            "hp_households": "{:,.3f}",
            "hp_household_share": "{:.6f}",
            "hp_stock_raw": "{:,.4f}",
            "hp_stock_floor": "{:,}",
            "fractional_remainder": "{:.4f}",
            "hp_stock_final": "{:,}",
        }
    )
)

National heating heat-pump stock: 407,637
Sum of federal-state shares     : 1.000000000000000
Sum of final federal-state stock: 407,637
Rounding adjustments (+1)       : 4



,federal_state_id,federal_state_code,hp_households,hp_household_share,hp_stock_raw,hp_stock_floor,fractional_remainder,rounding_adjustment,hp_stock_final
0,1,Burgenland,"39,639.147",0.064155,"26,152.1418","26,152",0.1418,False,"26,152"
1,2,Kärnten,"41,888.743",0.067796,"27,636.3251","27,636",0.3251,False,"27,636"
2,3,Niederösterreich,"173,941.831",0.281523,"114,759.0657","114,759",0.0657,False,"114,759"
3,4,Oberösterreich,"140,525.840",0.227439,"92,712.6844","92,712",0.6844,True,"92,713"
4,5,Salzburg,"32,716.061",0.052951,"21,584.5983","21,584",0.5983,True,"21,585"
5,6,Steiermark,"76,566.212",0.123921,"50,514.9731","50,514",0.9731,True,"50,515"
6,7,Tirol,"48,914.493",0.079167,"32,271.6016","32,271",0.6016,True,"32,272"
7,8,Vorarlberg,"38,895.402",0.062952,"25,661.4523","25,661",0.4523,False,"25,661"
8,9,Wien,"24,773.056",0.040095,"16,344.1578","16,344",0.1578,False,"16,344"


## 9. Energy data as validation only

`data/Energieeinsatz_HP_Bundesländer.xlsx` (Statistik Austria, STATcube:
*Energieeinsatz in Haushalten nach Bundesländern*, energy carrier
*Wärmepumpe*, survey period 2023/2024, sum of all end uses) is used **only**
as a validation and comparison source. It never enters the allocation and is
never used to correct the household-based distribution.

**Internal consistency of the household source**

$$
E_s^{HP} \;\approx\; H_s^{HP}\cdot \bar{e}_s^{HP}
$$

where $E_s^{HP}$ is the total heat-pump energy input of federal state $s$ in
GJ, $H_s^{HP}$ is the number of heat-pump households in $s$, and
$\bar{e}_s^{HP}$ is the average energy input per heat-pump household in $s$.

**Cross-source comparison** of the energy input reported by the two available
files, documenting absolute deviation, relative deviation, possible rounding
differences and conspicuous federal states.

**No automatic correction is derived from any of these checks.**

In [9]:
energy_validation_raw = pd.read_excel(
    HP_ENERGY_VALIDATION_PATH, sheet_name="Datenblatt 0", header=None, dtype=str
)

state_header_rows = [
    row for row in range(len(energy_validation_raw))
    if str(energy_validation_raw.iloc[row, 0]).strip() == "Bundesländer"
]
if len(state_header_rows) != 1:
    raise ValueError(
        "Expected exactly one 'Bundesländer' header row in the energy "
        f"validation workbook, found {len(state_header_rows)}."
    )
state_header_row = energy_validation_raw.iloc[state_header_rows[0]]

carrier_rows = [
    row for row in range(len(energy_validation_raw))
    if str(energy_validation_raw.iloc[row, 1]).strip() == HEAT_PUMP_ROW_LABEL
]
if len(carrier_rows) != 1:
    raise ValueError(
        f"Expected exactly one '{HEAT_PUMP_ROW_LABEL}' row in the energy "
        f"validation workbook, found {len(carrier_rows)}."
    )
carrier_row = energy_validation_raw.iloc[carrier_rows[0]]

cross_source_records = []
for state_id, state_name in GKZ_PREFIX_TO_STATE_NAME.items():
    column = locate_unique_column(state_header_row, state_name)
    value = parse_numeric(pd.Series([carrier_row.iloc[column]])).iloc[0]
    if pd.isna(value):
        raise ValueError(
            f"Energy validation workbook: value for {state_name} could not be parsed."
        )
    cross_source_records.append(
        {"federal_state_id": state_id,
         "federal_state_code": state_name,
         "energy_input_gj_validation_file": float(value)}
    )

energy_cross_source_validation = federal_state_audit[
    ["federal_state_id", "federal_state_code", "hp_households",
     "hp_energy_input_gj", "gj_per_hp_household"]
].merge(pd.DataFrame(cross_source_records), on=["federal_state_id", "federal_state_code"],
        how="outer", validate="one_to_one")

if energy_cross_source_validation.isna().any().any():
    raise ValueError(
        "Energy cross-source comparison is incomplete: a federal state is "
        "missing from one of the two files."
    )

# 1) Internal consistency of the household source: E ~= H * e_bar.
energy_cross_source_validation["implied_energy_gj_households_times_mean"] = (
    energy_cross_source_validation["hp_households"]
    * energy_cross_source_validation["gj_per_hp_household"]
)
energy_cross_source_validation["internal_absolute_deviation_gj"] = (
    energy_cross_source_validation["implied_energy_gj_households_times_mean"]
    - energy_cross_source_validation["hp_energy_input_gj"]
)
energy_cross_source_validation["internal_relative_deviation"] = (
    energy_cross_source_validation["internal_absolute_deviation_gj"]
    / energy_cross_source_validation["hp_energy_input_gj"]
)

# 2) Cross-source comparison of the two available files.
energy_cross_source_validation["cross_source_absolute_deviation_gj"] = (
    energy_cross_source_validation["energy_input_gj_validation_file"]
    - energy_cross_source_validation["hp_energy_input_gj"]
)
energy_cross_source_validation["cross_source_relative_deviation"] = (
    energy_cross_source_validation["cross_source_absolute_deviation_gj"]
    / energy_cross_source_validation["hp_energy_input_gj"]
)
# The validation file publishes whole GJ; the household file publishes full
# precision. A deviation of at most 0.5 GJ is therefore explainable purely by
# rounding to the nearest integer GJ.
energy_cross_source_validation["explained_by_integer_gj_rounding"] = (
    energy_cross_source_validation["cross_source_absolute_deviation_gj"].abs() <= 0.5
)

# 3) Informational only: does the household share differ from the energy share?
energy_cross_source_validation["hp_household_share"] = (
    energy_cross_source_validation["hp_households"]
    / energy_cross_source_validation["hp_households"].sum()
)
energy_cross_source_validation["hp_energy_share"] = (
    energy_cross_source_validation["hp_energy_input_gj"]
    / energy_cross_source_validation["hp_energy_input_gj"].sum()
)
energy_cross_source_validation["share_difference_energy_minus_households"] = (
    energy_cross_source_validation["hp_energy_share"]
    - energy_cross_source_validation["hp_household_share"]
)

INTERNAL_CONSISTENCY_TOLERANCE = 1e-4
CROSS_SOURCE_TOLERANCE = 1e-4

check_energy_internal_consistency = bool(
    (energy_cross_source_validation["internal_relative_deviation"].abs()
     <= INTERNAL_CONSISTENCY_TOLERANCE).all()
)
check_energy_cross_source_consistency = bool(
    (energy_cross_source_validation["cross_source_relative_deviation"].abs()
     <= CROSS_SOURCE_TOLERANCE).all()
)
check_cross_source_rounding_explained = bool(
    energy_cross_source_validation["explained_by_integer_gj_rounding"].all()
)

max_cross_source_state = energy_cross_source_validation.loc[
    energy_cross_source_validation["cross_source_absolute_deviation_gj"].abs().idxmax()
]
max_share_shift_state = energy_cross_source_validation.loc[
    energy_cross_source_validation["share_difference_energy_minus_households"].abs().idxmax()
]

print("E_s ~= H_s * e_bar_s holds for every federal state "
      f"(|rel. deviation| <= {INTERNAL_CONSISTENCY_TOLERANCE}):",
      check_energy_internal_consistency)
print("Both files report the same federal-state energy input "
      f"(|rel. deviation| <= {CROSS_SOURCE_TOLERANCE}):",
      check_energy_cross_source_consistency)
print("Every cross-source deviation is explained by rounding to whole GJ "
      "(|deviation| <= 0.5 GJ):", check_cross_source_rounding_explained)
print()
print("Largest absolute cross-source deviation: "
      f"{max_cross_source_state['federal_state_code']} "
      f"({max_cross_source_state['cross_source_absolute_deviation_gj']:+.4f} GJ, "
      f"{max_cross_source_state['cross_source_relative_deviation']:+.3e} relative)")
print()
print("Informational only - the energy share is NOT used for allocation.")
print("Largest difference between energy share and household share: "
      f"{max_share_shift_state['federal_state_code']} "
      f"({max_share_shift_state['share_difference_energy_minus_households']:+.4f} "
      "percentage points as a fraction)")
print("No automatic correction is applied on the basis of these checks.")
print()
display(
    energy_cross_source_validation[
        ["federal_state_id", "federal_state_code", "hp_energy_input_gj",
         "energy_input_gj_validation_file", "cross_source_absolute_deviation_gj",
         "cross_source_relative_deviation", "explained_by_integer_gj_rounding",
         "internal_relative_deviation", "hp_household_share", "hp_energy_share",
         "share_difference_energy_minus_households"]
    ]
)

E_s ~= H_s * e_bar_s holds for every federal state (|rel. deviation| <= 0.0001): True
Both files report the same federal-state energy input (|rel. deviation| <= 0.0001): True
Every cross-source deviation is explained by rounding to whole GJ (|deviation| <= 0.5 GJ): True

Largest absolute cross-source deviation: Wien (+0.4717 GJ, +9.204e-07 relative)

Informational only - the energy share is NOT used for allocation.
Largest difference between energy share and household share: Oberösterreich (+0.0358 percentage points as a fraction)
No automatic correction is applied on the basis of these checks.



,federal_state_id,federal_state_code,hp_energy_input_gj,energy_input_gj_validation_file,cross_source_absolute_deviation_gj,cross_source_relative_deviation,explained_by_integer_gj_rounding,internal_relative_deviation,hp_household_share,hp_energy_share,share_difference_energy_minus_households
0,1,Burgenland,8.810170e+05,881017.0,-0.049611,-5.631085e-08,True,1.585649e-15,0.064155,0.055586,-0.008570
1,2,Kärnten,1.092810e+06,1092810.0,-0.261483,-2.392757e-07,True,4.048079e-15,0.067796,0.068948,0.001152
2,3,Niederösterreich,4.321572e+06,4321572.0,-0.459715,-1.063767e-07,True,1.939549e-15,0.281523,0.272659,-0.008864
3,4,Oberösterreich,4.172924e+06,4172924.0,-0.041061,-9.839949e-09,True,5.579556e-16,0.227439,0.263280,0.035841
4,5,Salzburg,9.015926e+05,901593.0,0.441782,4.900018e-07,True,1.420341e-15,0.052951,0.056884,0.003933
5,6,Steiermark,1.869720e+06,1869720.0,0.392142,2.097332e-07,True,-2.988649e-15,0.123921,0.117965,-0.005956
6,7,Tirol,1.201818e+06,1201818.0,0.421129,3.504101e-07,True,-3.487178e-15,0.079167,0.075826,-0.003342
7,8,Vorarlberg,8.957361e+05,895736.0,-0.076127,-8.498801e-08,True,1.819525e-15,0.062952,0.056514,-0.006438
8,9,Wien,5.125615e+05,512562.0,0.471749,9.203762e-07,True,-2.271246e-16,0.040095,0.032339,-0.007756


## 10. Load the municipal building stock (Stage 2 proxy)

Source: `data/AEST_Gemeindeergebnisse_2024.ods` (Statistik Austria,
*Abgestimmte Erwerbsstatistik 2024 — Ausgewählte Ergebnisse*), sheet
`Gemeindetab_(Gebietsstand_2026)`. The territorial status 2026 matches the
territorial status of the canonical municipality list, so the two datasets are
joined on identical Gemeindekennziffern.

Columns read from the block *Gebäudebestand*:

| Source column | Read as |
|---|---|
| `Gebäude gesamt` | `buildings_total_all_types` |
| `Gebäude mit 1 oder 2 Wohnungen in %` | share of buildings with one or two dwellings |
| `Gebäude mit 3 oder mehr Wohnungen in %` | share of buildings with three or more dwellings |
| `Andere Gebäude1 in %` | share of other (non-residential) buildings — validation only |

The source publishes the building classes as **percentage shares rounded to
one decimal place**, not as absolute counts. The class counts are therefore
derived:

$$
B_m^{1}+B_m^{2} \;=\; B_m^{\text{total}}\cdot\frac{p_m^{1\text{-}2}}{100},
\qquad
B_m^{3+} \;=\; B_m^{\text{total}}\cdot\frac{p_m^{3+}}{100}
$$

$$
B_m^{res} \;=\; \left(B_m^{1}+B_m^{2}\right) + B_m^{3+}
$$

where $B_m^{\text{total}}$ is the total number of buildings of all types in
municipality $m$, $p_m^{1\text{-}2}$ and $p_m^{3+}$ are the published
percentage shares, $B_m^{1}$ and $B_m^{2}$ are residential buildings with one
and two dwellings, $B_m^{3+}$ are residential buildings with three or more
dwellings, and $B_m^{res}$ is the total number of residential buildings.

The source combines the one-dwelling and two-dwelling classes into a single
published class, so $B_m^{1}$ and $B_m^{2}$ are not separately available; only
their sum is. This is sufficient for both the main variant (all residential
buildings) and the sensitivity variant of Section 13 (one- and two-dwelling
buildings only). Because the shares are rounded to one decimal place, the
derived class counts carry a rounding uncertainty of up to
$\pm 0.05\,\% \cdot B_m^{\text{total}}$ per class. This is quantified against
the independent federal-state rows of the same source further below.

### Deterministic resolution of the municipality rows

The sheet mixes hierarchy levels in one identifier column: one-digit codes are
Austria and the federal states, three-digit codes are political districts
(including the 23 Viennese municipal districts), five-digit codes are
municipalities. Every municipality of the canonical master is resolved by
exactly one of three documented rules, and each rule is **re-derived and
validated from the master data itself** rather than asserted:

| Rule | Condition | Justification |
|---|---|---|
| `direct_municipality_row` | a five-digit AEST row with the same GKZ exists | ordinary case |
| `statutory_city_political_district_row` | no five-digit row exists, but a three-digit political-district row exists whose code is the GKZ prefix, **and** the master contains exactly one municipality with that prefix, whose GKZ is that prefix followed by `01` | an Austrian *Statutarstadt* is simultaneously a political district and a single municipality, so the district row covers exactly that one municipality's territory |
| `vienna_federal_state_row` | GKZ `90001`, resolved against the one-digit federal-state row `9`, **and** the master contains exactly one municipality in federal state 9 | Vienna is simultaneously a federal state and a single municipality; the 23 Viennese district rows are sub-municipal and are therefore not used |

Join key is the GKZ throughout. Municipality names are taken from the
canonical master only and are never overwritten by AEST labels; the differing
labels are reported as a diagnostic in Section 10a rather than silently
corrected. If any municipality cannot be resolved, or if the source contains a
five-digit code that is not in the canonical master, the affected records are
written to the diagnostic table and the workflow stops with an explicit
error.

In [10]:
BUILDINGS_SHEET = f"Gemeindetab_(Gebietsstand_{BUILDING_PROXY_TERRITORIAL_STATUS})"

buildings_raw = pd.read_excel(
    BUILDINGS_PATH, sheet_name=BUILDINGS_SHEET, engine="odf", header=None, dtype=str
)

group_header_rows = [
    row for row in range(min(10, len(buildings_raw)))
    if str(buildings_raw.iloc[row, 0]).strip() == "Gemeinde-kennziffer"
]
if len(group_header_rows) != 1:
    raise ValueError(
        "Expected exactly one 'Gemeinde-kennziffer' header row, "
        f"found {len(group_header_rows)}."
    )
group_header_row = group_header_rows[0]
label_header_row = group_header_row + 1
first_data_row = label_header_row + 1

label_header = buildings_raw.iloc[label_header_row]
group_header = buildings_raw.iloc[group_header_row]

column_total_buildings = locate_unique_column(label_header, "Gebäude gesamt")
column_share_1_2 = locate_unique_column(
    label_header, "Gebäude mit 1 oder 2 Wohnungen in %"
)
column_share_3_plus = locate_unique_column(
    label_header, "Gebäude mit 3 oder mehr Wohnungen in %"
)
other_building_columns = [
    position for position, value in enumerate(label_header.tolist())
    if str(value).strip().startswith("Andere Gebäude")
]
if len(other_building_columns) != 1:
    raise ValueError(
        "Expected exactly one 'Andere Gebäude ... in %' column, "
        f"found {len(other_building_columns)}."
    )
column_share_other = other_building_columns[0]

# Guard: the four columns must sit under the 'Gebäudebestand' block header.
if str(group_header.iloc[column_total_buildings]).strip() != "Gebäudebestand":
    raise ValueError(
        "'Gebäude gesamt' is not located under the 'Gebäudebestand' block header."
    )

records = buildings_raw.iloc[first_data_row:].copy()
records["aest_code"] = records[0].astype(str).str.strip()
records["aest_label"] = records[1].astype(str).str.strip()

numeric_code_mask = records["aest_code"].str.fullmatch(r"\d+")
non_data_rows = records.loc[~numeric_code_mask].copy()
records = records.loc[numeric_code_mask].copy()

# Nothing is dropped silently: every excluded source row is recorded.
aest_row_usage_rows = [
    {
        "aest_code": row["aest_code"][:60],
        "aest_label": row["aest_label"][:60],
        "code_length": len(row["aest_code"]),
        "usage": "excluded_non_numeric_code",
        "reason": "footnote, source note or blank row - not a territorial unit",
    }
    for _, row in non_data_rows.iterrows()
]

records["buildings_total_all_types"] = parse_numeric(records[column_total_buildings])
records["share_1_2_dwellings_pct"] = parse_numeric(records[column_share_1_2])
records["share_3_plus_dwellings_pct"] = parse_numeric(records[column_share_3_plus])
records["share_other_buildings_pct"] = parse_numeric(records[column_share_other])

unparseable = records[
    records[
        ["buildings_total_all_types", "share_1_2_dwellings_pct",
         "share_3_plus_dwellings_pct", "share_other_buildings_pct"]
    ].isna().any(axis=1)
]
if len(unparseable) > 0:
    display(unparseable[["aest_code", "aest_label"]])
    raise ValueError(
        f"{len(unparseable)} AEST row(s) contain building values that could not "
        "be parsed as numbers. No value is imputed or dropped silently - resolve "
        "the source data before continuing."
    )

records["buildings_1_2_dwellings"] = (
    records["buildings_total_all_types"] * records["share_1_2_dwellings_pct"] / 100.0
)
records["buildings_3_plus_dwellings"] = (
    records["buildings_total_all_types"] * records["share_3_plus_dwellings_pct"] / 100.0
)
records["residential_buildings"] = (
    records["buildings_1_2_dwellings"] + records["buildings_3_plus_dwellings"]
)

check_no_negative_buildings = bool(
    (records["buildings_total_all_types"] >= 0).all()
    and (records["buildings_1_2_dwellings"] >= 0).all()
    and (records["buildings_3_plus_dwellings"] >= 0).all()
)
assert check_no_negative_buildings, "Negative building count found in the source."

published_share_sum = (
    records["share_1_2_dwellings_pct"]
    + records["share_3_plus_dwellings_pct"]
    + records["share_other_buildings_pct"]
)
max_share_sum_deviation = float((published_share_sum - 100.0).abs().max())
check_building_shares_sum_to_100 = max_share_sum_deviation <= 0.35

records["code_length"] = records["aest_code"].str.len()
aest_municipality_rows = records[records["code_length"] == 5].set_index("aest_code")
aest_district_rows = records[records["code_length"] == 3].set_index("aest_code")
aest_state_rows = records[records["code_length"] == 1].set_index("aest_code")

if not aest_municipality_rows.index.is_unique:
    raise ValueError("Duplicate five-digit Gemeindekennziffer in the AEST source.")
if not aest_district_rows.index.is_unique:
    raise ValueError("Duplicate three-digit district code in the AEST source.")
if not aest_state_rows.index.is_unique:
    raise ValueError("Duplicate one-digit territorial code in the AEST source.")

print("AEST sheet:", BUILDINGS_SHEET)
print("Territorial rows read:", len(records),
      f"(one-digit {len(aest_state_rows)}, three-digit {len(aest_district_rows)}, "
      f"five-digit {len(aest_municipality_rows)})")
print("Non-numeric rows excluded and recorded:", len(non_data_rows))
print("No negative building counts:", check_no_negative_buildings)
print(f"Published class shares sum to 100% (max deviation "
      f"{max_share_sum_deviation:.2f} pp, from one-decimal rounding):",
      check_building_shares_sum_to_100)

AEST sheet: Gemeindetab_(Gebietsstand_2026)
Territorial rows read: 2203 (one-digit 10, three-digit 116, five-digit 2077)
Non-numeric rows excluded and recorded: 1
No negative building counts: True
Published class shares sum to 100% (max deviation 0.10 pp, from one-decimal rounding): True


### 10a. Resolve, validate and diagnose the municipal building proxy

In [11]:
master_prefix_counts = municipalities["municipality_id"].str[:3].value_counts()
master_state_counts = municipalities["federal_state_id"].value_counts()

proxy_rows = []
unmatched_records_rows = []
special_rule_rows = []

for record in municipalities.itertuples(index=False):
    municipality_id = record.municipality_id
    prefix = municipality_id[:3]

    if municipality_id in aest_municipality_rows.index:
        source = aest_municipality_rows.loc[municipality_id]
        rule = "direct_municipality_row"
        source_code = municipality_id

    elif municipality_id == "90001":
        # Vienna: simultaneously a federal state and a single municipality.
        vienna_state_code = "9"
        if vienna_state_code not in aest_state_rows.index:
            unmatched_records_rows.append({
                "record_type": "municipality_without_building_data",
                "identifier": municipality_id,
                "name": record.municipality_name,
                "issue": "no federal-state row '9' available for Vienna",
                "source_file": BUILDINGS_PATH.name,
            })
            continue
        if int(master_state_counts.get(vienna_state_code, 0)) != 1:
            raise ValueError(
                "Vienna rule is not applicable: federal state 9 contains "
                f"{int(master_state_counts.get(vienna_state_code, 0))} municipalities "
                "in the canonical master, expected exactly 1."
            )
        source = aest_state_rows.loc[vienna_state_code]
        rule = "vienna_federal_state_row"
        source_code = vienna_state_code
        special_rule_rows.append({
            "resolution_rule": rule,
            "municipality_id": municipality_id,
            "municipality_name": record.municipality_name,
            "aest_source_code": source_code,
            "aest_source_label": source["aest_label"],
            "municipalities_in_master_under_this_code": int(
                master_state_counts.get(vienna_state_code, 0)
            ),
            "evidence": (
                "Vienna is simultaneously a federal state and a single "
                "municipality; the 23 three-digit Viennese district rows are "
                "sub-municipal and are not used."
            ),
        })

    elif prefix in aest_district_rows.index:
        municipalities_under_prefix = int(master_prefix_counts.get(prefix, 0))
        if municipalities_under_prefix != 1:
            unmatched_records_rows.append({
                "record_type": "municipality_without_building_data",
                "identifier": municipality_id,
                "name": record.municipality_name,
                "issue": (
                    "no five-digit AEST row; the three-digit district row "
                    f"{prefix} covers {municipalities_under_prefix} municipalities "
                    "and therefore cannot be attributed to a single municipality"
                ),
                "source_file": BUILDINGS_PATH.name,
            })
            continue
        if municipality_id != prefix + "01":
            unmatched_records_rows.append({
                "record_type": "municipality_without_building_data",
                "identifier": municipality_id,
                "name": record.municipality_name,
                "issue": (
                    f"sole municipality of district {prefix} but its GKZ is not "
                    f"{prefix}01, so the statutory-city rule does not apply"
                ),
                "source_file": BUILDINGS_PATH.name,
            })
            continue
        source = aest_district_rows.loc[prefix]
        rule = "statutory_city_political_district_row"
        source_code = prefix
        special_rule_rows.append({
            "resolution_rule": rule,
            "municipality_id": municipality_id,
            "municipality_name": record.municipality_name,
            "aest_source_code": source_code,
            "aest_source_label": source["aest_label"],
            "municipalities_in_master_under_this_code": municipalities_under_prefix,
            "evidence": (
                "statutory city: political district with exactly one municipality "
                "in the canonical master, GKZ = district code + '01'"
            ),
        })

    else:
        unmatched_records_rows.append({
            "record_type": "municipality_without_building_data",
            "identifier": municipality_id,
            "name": record.municipality_name,
            "issue": "no AEST municipality, district or federal-state row found",
            "source_file": BUILDINGS_PATH.name,
        })
        continue

    proxy_rows.append({
        "municipality_id": municipality_id,
        "buildings_total_all_types": float(source["buildings_total_all_types"]),
        "buildings_1_2_dwellings": float(source["buildings_1_2_dwellings"]),
        "buildings_3_plus_dwellings": float(source["buildings_3_plus_dwellings"]),
        "residential_buildings": float(source["residential_buildings"]),
        "building_proxy_resolution_rule": rule,
        "aest_source_code": source_code,
        "aest_source_label": source["aest_label"],
    })

building_proxy = pd.DataFrame(proxy_rows)
building_proxy_special_rows = pd.DataFrame(special_rule_rows)

# Five-digit AEST codes that are not in the canonical master would be unknown
# municipalities and must not be introduced silently.
unknown_aest_municipalities = sorted(
    set(aest_municipality_rows.index) - CANONICAL_MUNICIPALITY_IDS
)
for code in unknown_aest_municipalities:
    unmatched_records_rows.append({
        "record_type": "aest_municipality_not_in_canonical_master",
        "identifier": code,
        "name": str(aest_municipality_rows.loc[code, "aest_label"]),
        "issue": "five-digit AEST code is not part of the canonical municipality list",
        "source_file": BUILDINGS_PATH.name,
    })

unmatched_records = pd.DataFrame(
    unmatched_records_rows,
    columns=["record_type", "identifier", "name", "issue", "source_file"],
)

# Aggregate rows of the source that are structurally not municipality rows are
# recorded as deliberately unused rather than dropped without trace.
used_source_codes = set(building_proxy["aest_source_code"]) if len(building_proxy) else set()
for code, row in records.set_index("aest_code").iterrows():
    if code in used_source_codes:
        usage, reason = "used_as_municipal_proxy", "resolved to exactly one municipality"
    elif len(code) == 5:
        usage, reason = "unused_five_digit_row", "not part of the canonical municipality list"
    elif len(code) == 3:
        usage, reason = ("unused_aggregate_row",
                         "political district aggregate (or Viennese municipal district, "
                         "which is sub-municipal)")
    else:
        usage, reason = ("validation_only_aggregate_row",
                         "Austria / federal-state aggregate, used only for the "
                         "state-level plausibility check")
    aest_row_usage_rows.append({
        "aest_code": code,
        "aest_label": str(row["aest_label"])[:60],
        "code_length": len(code),
        "usage": usage,
        "reason": reason,
    })
aest_row_usage = pd.DataFrame(aest_row_usage_rows)

check_no_unmatched_records = len(unmatched_records) == 0
check_all_municipalities_have_building_data = (
    len(building_proxy) == len(municipalities)
)
check_no_unknown_aest_municipalities = len(unknown_aest_municipalities) == 0

building_proxy_rule_summary = (
    building_proxy["building_proxy_resolution_rule"]
    .value_counts()
    .rename_axis("resolution_rule")
    .rename("number_of_municipalities")
    .reset_index()
    .sort_values("resolution_rule")
    .reset_index(drop=True)
) if len(building_proxy) else pd.DataFrame(
    columns=["resolution_rule", "number_of_municipalities"]
)

print("Municipalities in the canonical master :", len(municipalities))
print("Municipalities with a building proxy   :", len(building_proxy))
print("Unmatched / unresolvable records       :", len(unmatched_records))
print("Unknown five-digit AEST codes          :", len(unknown_aest_municipalities))
print()
display(building_proxy_rule_summary)
if len(building_proxy_special_rows):
    print("Municipalities resolved by a documented aggregate-row rule:")
    display(building_proxy_special_rows)

if not check_no_unmatched_records:
    display(unmatched_records)
    raise ValueError(
        f"{len(unmatched_records)} record(s) could not be matched between the "
        "canonical municipality list and the municipal building dataset. "
        "The municipality-level disaggregation is not computed: an incomplete "
        "or ambiguous proxy would silently redistribute a federal state's "
        "entire stock over the remaining municipalities. Resolve the records "
        "listed above before continuing."
    )

Municipalities in the canonical master : 2092
Municipalities with a building proxy   : 2092
Unmatched / unresolvable records       : 0
Unknown five-digit AEST codes          : 0



,resolution_rule,number_of_municipalities
0,direct_municipality_row,2077
1,statutory_city_political_district_row,14
2,vienna_federal_state_row,1


Municipalities resolved by a documented aggregate-row rule:


,resolution_rule,municipality_id,municipality_name,aest_source_code,aest_source_label,municipalities_in_master_under_this_code,evidence
0,statutory_city_political_district_row,10101,Eisenstadt,101,Eisenstadt (Stadt),1,statutory city: political district with exactl...
1,statutory_city_political_district_row,10201,Rust,102,Rust (Stadt),1,statutory city: political district with exactl...
2,statutory_city_political_district_row,20101,Klagenfurt am Wörthersee,201,Klagenfurt Stadt,1,statutory city: political district with exactl...
3,statutory_city_political_district_row,20201,Villach,202,Villach Stadt,1,statutory city: political district with exactl...
4,statutory_city_political_district_row,30101,Krems an der Donau,301,Krems an der Donau (Stadt),1,statutory city: political district with exactl...
5,statutory_city_political_district_row,30201,St. Pölten,302,Sankt Pölten (Stadt),1,statutory city: political district with exactl...
6,statutory_city_political_district_row,30301,Waidhofen an der Ybbs,303,Waidhofen an der Ybbs (Stadt),1,statutory city: political district with exactl...
7,statutory_city_political_district_row,30401,Wiener Neustadt,304,Wiener Neustadt (Stadt),1,statutory city: political district with exactl...
8,statutory_city_political_district_row,40101,Linz,401,Stadt Linz,1,statutory city: political district with exactl...
9,statutory_city_political_district_row,40201,Steyr,402,Stadt Steyr,1,statutory city: political district with exactl...


### 10b. Building-proxy plausibility and name diagnostics

In [12]:
municipality_building_proxy = municipalities.merge(
    building_proxy, on="municipality_id", how="left", validate="one_to_one"
)
assert len(municipality_building_proxy) == len(municipalities), (
    "The building-proxy join changed the number of municipality rows."
)
assert municipality_building_proxy["residential_buildings"].notna().all(), (
    "A municipality has no residential-building value after the join."
)

# Plausibility: the municipality-level values must reconstruct the independent
# federal-state rows of the same source. Residual deviations come from the
# one-decimal rounding of the published class shares.
state_building_sums = (
    municipality_building_proxy.groupby("federal_state_id")["residential_buildings"]
    .sum()
    .rename("municipal_sum_residential_buildings")
    .reset_index()
)
state_building_reference = pd.DataFrame(
    {
        "federal_state_id": aest_state_rows.index,
        "aest_state_row_residential_buildings": aest_state_rows["residential_buildings"].to_numpy(),
    }
)
building_proxy_state_validation = state_building_sums.merge(
    state_building_reference, on="federal_state_id", how="left", validate="one_to_one"
)
building_proxy_state_validation["federal_state_code"] = (
    building_proxy_state_validation["federal_state_id"].map(GKZ_PREFIX_TO_STATE_NAME)
)
building_proxy_state_validation["absolute_deviation"] = (
    building_proxy_state_validation["municipal_sum_residential_buildings"]
    - building_proxy_state_validation["aest_state_row_residential_buildings"]
)
building_proxy_state_validation["relative_deviation"] = (
    building_proxy_state_validation["absolute_deviation"]
    / building_proxy_state_validation["aest_state_row_residential_buildings"]
)
# Vienna is the one state whose single municipality takes its proxy from this
# very federal-state row, so its deviation is zero by construction and is not
# an independent confirmation. Flagged rather than quietly presented as one.
building_proxy_state_validation["independent_reconstruction"] = (
    building_proxy_state_validation["federal_state_id"] != "9"
)

BUILDING_STATE_TOLERANCE = 0.01
max_building_state_deviation = float(
    building_proxy_state_validation["relative_deviation"].abs().max()
)
check_building_proxy_reconstructs_state_rows = (
    max_building_state_deviation <= BUILDING_STATE_TOLERANCE
)

# No silent name correction: names come from the canonical master only. Where
# the AEST label differs, that is reported, not applied.
municipality_name_comparison = municipality_building_proxy.loc[
    municipality_building_proxy["municipality_name"].str.strip()
    != municipality_building_proxy["aest_source_label"].str.strip(),
    ["municipality_id", "municipality_name", "aest_source_label",
     "building_proxy_resolution_rule"],
].rename(columns={"municipality_name": "canonical_master_name"}).reset_index(drop=True)
municipality_name_comparison["applied_name"] = "canonical_master_name"
municipality_name_comparison["note"] = (
    "labels differ; the join uses the GKZ only and the canonical name is kept"
)

print(f"Federal-state plausibility of the municipal building proxy "
      f"(tolerance {BUILDING_STATE_TOLERANCE:.0%}):",
      check_building_proxy_reconstructs_state_rows)
print(f"Largest relative deviation: {max_building_state_deviation:.4%}")
print()
display(
    building_proxy_state_validation[
        ["federal_state_id", "federal_state_code",
         "municipal_sum_residential_buildings",
         "aest_state_row_residential_buildings",
         "absolute_deviation", "relative_deviation"]
    ]
)
print(f"Municipalities whose AEST label differs from the canonical master name: "
      f"{len(municipality_name_comparison)} (reported, never applied)")
display(municipality_name_comparison.head(10))

assert check_building_proxy_reconstructs_state_rows, (
    "The municipal building proxy does not reconstruct the federal-state rows "
    "of its own source within tolerance - the row resolution is not trustworthy."
)

Federal-state plausibility of the municipal building proxy (tolerance 1%): True
Largest relative deviation: 0.0774%



,federal_state_id,federal_state_code,municipal_sum_residential_buildings,aest_state_row_residential_buildings,absolute_deviation,relative_deviation
0,1,Burgenland,125998.155,126003.150,-4.995,-0.000040
1,2,Kärnten,166650.199,166710.261,-60.062,-0.000360
2,3,Niederösterreich,587243.553,586789.593,453.960,0.000774
3,4,Oberösterreich,385259.447,385285.571,-26.124,-0.000068
4,5,Salzburg,121141.611,121137.093,4.518,0.000037
5,6,Steiermark,345190.049,345148.650,41.399,0.000120
6,7,Tirol,168217.087,168200.028,17.059,0.000101
7,8,Vorarlberg,96680.095,96649.520,30.575,0.000316
8,9,Wien,161512.595,161512.595,0.000,0.000000


Municipalities whose AEST label differs from the canonical master name: 25 (reported, never applied)


,municipality_id,canonical_master_name,aest_source_label,building_proxy_resolution_rule,applied_name,note
0,10101,Eisenstadt,Eisenstadt (Stadt),statutory_city_political_district_row,canonical_master_name,labels differ; the join uses the GKZ only and ...
1,10201,Rust,Rust (Stadt),statutory_city_political_district_row,canonical_master_name,labels differ; the join uses the GKZ only and ...
2,20101,Klagenfurt am Wörthersee,Klagenfurt Stadt,statutory_city_political_district_row,canonical_master_name,labels differ; the join uses the GKZ only and ...
3,20201,Villach,Villach Stadt,statutory_city_political_district_row,canonical_master_name,labels differ; the join uses the GKZ only and ...
4,30101,Krems an der Donau,Krems an der Donau (Stadt),statutory_city_political_district_row,canonical_master_name,labels differ; the join uses the GKZ only and ...
5,30201,St. Pölten,Sankt Pölten (Stadt),statutory_city_political_district_row,canonical_master_name,labels differ; the join uses the GKZ only and ...
6,30301,Waidhofen an der Ybbs,Waidhofen an der Ybbs (Stadt),statutory_city_political_district_row,canonical_master_name,labels differ; the join uses the GKZ only and ...
7,30401,Wiener Neustadt,Wiener Neustadt (Stadt),statutory_city_political_district_row,canonical_master_name,labels differ; the join uses the GKZ only and ...
8,40101,Linz,Stadt Linz,statutory_city_political_district_row,canonical_master_name,labels differ; the join uses the GKZ only and ...
9,40201,Steyr,Stadt Steyr,statutory_city_political_district_row,canonical_master_name,labels differ; the join uses the GKZ only and ...


## 11. Stage 2 — federal states to municipalities

**Municipal proxy value (main variant)**

$$
X_m^{HP} \;=\; B_m^{res}
$$

where $X_m^{HP}$ is the municipal proxy value of municipality $m$ and
$B_m^{res}$ is its number of residential buildings. Within a federal state the
heat-pump stock is therefore distributed across municipalities in proportion
to the number of residential buildings. The main variant deliberately uses
neither population, nor municipal area, nor any freely chosen multi-family
weighting factor.

**Share of a municipality in the proxy total of its federal state**

$$
q_{m\mid s}^{HP} \;=\; \frac{X_m^{HP}}{\sum_{j\in\mathcal{M}_s} X_j^{HP}}
$$

where $m$ is the municipality under consideration, $s$ is its federal state,
$j$ is the running index over all municipalities of that federal state,
$\mathcal{M}_s$ is the set of municipalities of federal state $s$, and
$q_{m\mid s}^{HP}$ is the share of municipality $m$ in the proxy total of its
federal state.

**Unrounded modelled municipal stock**

$$
\widetilde{N}_m^{HP} \;=\; N_s^{HP}\cdot q_{m\mid s}^{HP}
$$

where $N_s^{HP}$ is the final federal-state stock from Stage 1,
$q_{m\mid s}^{HP}$ is the municipal proxy share, and $\widetilde{N}_m^{HP}$ is
the unrounded municipal stock.

**Integer allocation**, again with the Largest Remainder method and applied
**separately within each federal state**. Tie-break order:

1. decimal remainder, descending;
2. municipal proxy value, descending;
3. Gemeindekennziffer, ascending.

After rounding the following holds exactly for every federal state:

$$
\sum_{m\in\mathcal{M}_s} N_m^{HP} \;=\; N_s^{HP}
$$

and for Austria as a whole:

$$
\sum_{s\in\mathcal{S}}\sum_{m\in\mathcal{M}_s} N_m^{HP} \;=\; 407\,637
$$

Vienna is processed by exactly the same generic logic; there is no Vienna
special case. Because Vienna is a single municipality in the territorial
status used here, $q_{m\mid s}^{HP}=1$ and that municipality automatically
receives the entire Viennese federal-state stock.

In [13]:
def allocate_within_states(
    frame: pd.DataFrame, proxy_column: str, state_stock: dict
) -> pd.DataFrame:
    """Distribute each federal-state stock over its municipalities.

    The same generic routine is used for the main variant and for the
    sensitivity variant; there is no second, parallel implementation and no
    special case for any individual federal state or municipality.
    """
    allocated = []
    for state_id in sorted(frame["federal_state_id"].unique()):
        group = (
            frame.loc[frame["federal_state_id"] == state_id]
            .sort_values("municipality_id")
            .set_index("municipality_id")
            .copy()
        )
        proxy = group[proxy_column].astype(float)
        proxy_total = float(proxy.sum())
        if not proxy_total > 0:
            raise ValueError(
                f"Federal state {state_id} has a proxy total of {proxy_total}. "
                "A zero proxy total makes the municipal shares undefined."
            )
        stock = int(state_stock[state_id])

        share = proxy / proxy_total
        raw = stock * share
        floor_values = np.floor(raw).astype("int64")
        final = largest_remainder_apportionment(raw, stock, secondary_key=proxy)

        group["state_hp_stock_final"] = stock
        group["municipal_proxy_value"] = proxy
        group["state_proxy_total"] = proxy_total
        group["proxy_share_within_state"] = share
        group["hp_stock_raw"] = raw
        group["hp_stock_floor"] = floor_values
        group["fractional_remainder"] = raw - floor_values
        group["unit_count"] = final.astype("int64")
        group["rounding_adjustment"] = (final - floor_values).astype(bool)

        assert int(group["unit_count"].sum()) == stock, (
            f"Mass preservation failed for federal state {state_id}."
        )
        allocated.append(group.reset_index())

    return pd.concat(allocated, ignore_index=True).sort_values(
        "municipality_id"
    ).reset_index(drop=True)


municipality_result = allocate_within_states(
    municipality_building_proxy, "residential_buildings", STATE_STOCK
)
municipality_result["proxy_variant"] = "all_residential_buildings"
municipality_result["derivation_method"] = DERIVATION_METHOD
municipality_result["data_quality"] = DATA_QUALITY

state_proxy_totals = (
    municipality_result.groupby("federal_state_id")["municipal_proxy_value"].sum()
)
check_no_zero_state_proxy_total = bool((state_proxy_totals > 0).all())
check_row_count_preserved = len(municipality_result) == len(municipalities)
check_municipality_ids_unchanged = (
    set(municipality_result["municipality_id"]) == CANONICAL_MUNICIPALITY_IDS
)
check_municipality_id_unique_in_output = bool(
    municipality_result["municipality_id"].is_unique
)
check_unit_count_integer = bool(
    pd.api.types.is_integer_dtype(municipality_result["unit_count"])
)
check_unit_count_non_negative = bool((municipality_result["unit_count"] >= 0).all())
check_unit_count_not_missing = bool(municipality_result["unit_count"].notna().all())

state_sum_check = (
    municipality_result.groupby("federal_state_id")["unit_count"].sum().astype("int64")
)
check_state_sums_match = bool(
    all(int(state_sum_check[state_id]) == int(stock)
        for state_id, stock in STATE_STOCK.items())
)
national_municipal_sum = int(municipality_result["unit_count"].sum())
check_national_municipal_sum = national_municipal_sum == NATIONAL_HEATING_HP_STOCK

assert check_row_count_preserved, "The municipality row count changed during allocation."
assert check_municipality_ids_unchanged, "The set of municipality ids changed."
assert check_state_sums_match, "A federal-state municipal sum does not match its stock."
assert check_national_municipal_sum, "The national municipal sum is not 407,637."

print(f"Municipalities processed          : {len(municipality_result):,}")
print(f"National sum of unit_count        : {national_municipal_sum:,}")
print(f"Equals NATIONAL_HEATING_HP_STOCK  : {check_national_municipal_sum}")
print(f"Municipalities with unit_count = 0: "
      f"{int((municipality_result['unit_count'] == 0).sum()):,}")
print(f"Rounding adjustments (+1)         : "
      f"{int(municipality_result['rounding_adjustment'].sum()):,}")
print()
display(
    municipality_result.groupby(["federal_state_id", "federal_state_code"])
    .agg(
        municipalities=("municipality_id", "size"),
        state_stock=("state_hp_stock_final", "first"),
        municipal_sum=("unit_count", "sum"),
        min_unit_count=("unit_count", "min"),
        max_unit_count=("unit_count", "max"),
    )
    .reset_index()
)
print("Largest municipal values (main variant):")
display(
    municipality_result.nlargest(10, "unit_count")[
        ["municipality_id", "municipality_name", "federal_state_code",
         "residential_buildings", "proxy_share_within_state",
         "hp_stock_raw", "unit_count"]
    ]
)

Municipalities processed          : 2,092
National sum of unit_count        : 407,637
Equals NATIONAL_HEATING_HP_STOCK  : True
Municipalities with unit_count = 0: 0
Rounding adjustments (+1)         : 1,059



,federal_state_id,federal_state_code,municipalities,state_stock,municipal_sum,min_unit_count,max_unit_count
0,1,Burgenland,171,26152,26152,10,766
1,2,Kärnten,132,27636,27636,36,3089
2,3,Niederösterreich,573,114759,114759,8,2305
3,4,Oberösterreich,438,92713,92713,23,4762
4,5,Salzburg,119,21585,21585,11,3295
5,6,Steiermark,285,50515,50515,28,5485
6,7,Tirol,277,32272,32272,3,2052
7,8,Vorarlberg,96,25661,25661,14,2719
8,9,Wien,1,16344,16344,16344,16344


Largest municipal values (main variant):


,municipality_id,municipality_name,federal_state_code,residential_buildings,proxy_share_within_state,hp_stock_raw,unit_count
2091,90001,Wien,Wien,161512.595,1.000000,16344.000000,16344
1433,60101,Graz,Steiermark,37481.346,0.108582,5485.008037,5485
876,40101,Linz,Oberösterreich,19788.828,0.051365,4762.197591,4762
1314,50101,Salzburg,Salzburg,18493.688,0.152662,3295.203458,3295
171,20101,Klagenfurt am Wörthersee,Kärnten,18624.848,0.111760,3088.602968,3089
2064,80301,Dornbirn,Vorarlberg,10244.760,0.105966,2719.182127,2719
304,30201,St. Pölten,Niederösterreich,11792.895,0.020082,2304.564827,2305
878,40301,Wels,Oberösterreich,9246.226,0.024000,2225.111825,2225
801,32144,Klosterneuburg,Niederösterreich,10692.210,0.018207,2089.469218,2089
1718,70101,Innsbruck,Tirol,10694.316,0.063574,2051.676034,2052


## 12. Sensitivity variant — one- and two-dwelling buildings only

The published building classes allow a second, restricted proxy:

$$
X_{m,1\text{-}2}^{HP} \;=\; B_m^{1}+B_m^{2}
$$

where $B_m^{1}$ and $B_m^{2}$ are the residential buildings in municipality $m$
with one and with two dwellings. This variant uses only one- and two-dwelling
residential buildings and drops $B_m^{3+}$ entirely.

Its **sole purpose is to examine how sensitive the municipal results are to
the choice of building proxy.** The canonical main output continues to be based
on all residential buildings.

No weighted expression of the form

$$
B_m^{1}+B_m^{2}+\lambda B_m^{3+}
$$

is used, because no evidence-based source for $\lambda$ is available in this
project. Inventing a value for $\lambda$ would introduce an unsupported
weighting factor into the canonical result, so the analysis is restricted to
the two unweighted end members: all residential buildings (main) and
one-/two-dwelling buildings only (sensitivity).

Both variants distribute the identical federal-state stocks $N_s^{HP}$, so
every federal-state sum is identical by construction. The comparison therefore
measures redistribution **within** federal states, not between them.

In [14]:
sensitivity_result = allocate_within_states(
    municipality_building_proxy, "buildings_1_2_dwellings", STATE_STOCK
)
sensitivity_result["proxy_variant"] = "one_and_two_dwelling_buildings_only"

sensitivity_comparison = municipality_result[
    ["municipality_id", "municipality_name", "federal_state_id", "federal_state_code",
     "residential_buildings", "buildings_1_2_dwellings", "buildings_3_plus_dwellings",
     "unit_count"]
].rename(columns={"unit_count": "unit_count_main"}).merge(
    sensitivity_result[["municipality_id", "municipal_proxy_value",
                        "proxy_share_within_state", "hp_stock_raw", "unit_count"]]
    .rename(columns={
        "municipal_proxy_value": "sensitivity_proxy_value",
        "proxy_share_within_state": "sensitivity_proxy_share_within_state",
        "hp_stock_raw": "sensitivity_hp_stock_raw",
        "unit_count": "unit_count_sensitivity",
    }),
    on="municipality_id",
    how="inner",
    validate="one_to_one",
)
assert len(sensitivity_comparison) == len(municipalities), (
    "The sensitivity comparison lost or duplicated municipalities."
)

sensitivity_comparison["absolute_difference"] = (
    sensitivity_comparison["unit_count_sensitivity"]
    - sensitivity_comparison["unit_count_main"]
)
# Relative difference is undefined where the main variant allocates zero
# units. Those municipalities are reported as missing, never imputed as 0.
sensitivity_comparison["relative_difference"] = np.where(
    sensitivity_comparison["unit_count_main"] > 0,
    sensitivity_comparison["absolute_difference"]
    / sensitivity_comparison["unit_count_main"].replace(0, np.nan),
    np.nan,
)

undefined_relative = int(sensitivity_comparison["relative_difference"].isna().sum())

sensitivity_by_state = (
    sensitivity_comparison.groupby(["federal_state_id", "federal_state_code"])
    .apply(
        lambda group: pd.Series({
            "municipalities": len(group),
            "state_stock_main": int(group["unit_count_main"].sum()),
            "state_stock_sensitivity": int(group["unit_count_sensitivity"].sum()),
            "municipalities_changed": int((group["absolute_difference"] != 0).sum()),
            "share_of_municipalities_changed": float(
                (group["absolute_difference"] != 0).mean()
            ),
            "sum_absolute_difference": int(group["absolute_difference"].abs().sum()),
            "mean_absolute_difference": float(group["absolute_difference"].abs().mean()),
            "median_absolute_difference": float(
                group["absolute_difference"].abs().median()
            ),
            "max_positive_difference": int(group["absolute_difference"].max()),
            "max_negative_difference": int(group["absolute_difference"].min()),
            "mean_absolute_relative_difference": float(
                group["relative_difference"].abs().mean()
            ),
            "redistributed_share_of_state_stock": float(
                group["absolute_difference"].abs().sum()
                / (2 * max(1, int(group["unit_count_main"].sum())))
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

top_positive = sensitivity_comparison.nlargest(20, "absolute_difference").copy()
top_positive["deviation_direction"] = "largest_positive"
top_negative = sensitivity_comparison.nsmallest(20, "absolute_difference").copy()
top_negative["deviation_direction"] = "largest_negative"
sensitivity_top_deviations = pd.concat([top_positive, top_negative], ignore_index=True)[
    ["deviation_direction", "municipality_id", "municipality_name",
     "federal_state_code", "residential_buildings", "buildings_1_2_dwellings",
     "buildings_3_plus_dwellings", "unit_count_main", "unit_count_sensitivity",
     "absolute_difference", "relative_difference"]
]

check_sensitivity_state_sums_preserved = bool(
    (sensitivity_by_state["state_stock_main"]
     == sensitivity_by_state["state_stock_sensitivity"]).all()
)
check_sensitivity_national_sum = (
    int(sensitivity_result["unit_count"].sum()) == NATIONAL_HEATING_HP_STOCK
)

print("Sensitivity variant: one- and two-dwelling residential buildings only")
print(f"National sum (sensitivity)   : {int(sensitivity_result['unit_count'].sum()):,}")
print(f"Federal-state sums identical : {check_sensitivity_state_sums_preserved}")
print(f"Municipalities changed       : "
      f"{int((sensitivity_comparison['absolute_difference'] != 0).sum()):,} of "
      f"{len(sensitivity_comparison):,} "
      f"({(sensitivity_comparison['absolute_difference'] != 0).mean():.1%})")
print(f"Total units redistributed    : "
      f"{int(sensitivity_comparison['absolute_difference'].abs().sum()) // 2:,} "
      f"({sensitivity_comparison['absolute_difference'].abs().sum() / (2 * NATIONAL_HEATING_HP_STOCK):.2%} "
      "of the national stock)")
print(f"Largest positive deviation   : "
      f"{int(sensitivity_comparison['absolute_difference'].max()):+,}")
print(f"Largest negative deviation   : "
      f"{int(sensitivity_comparison['absolute_difference'].min()):+,}")
print(f"Relative difference undefined (main variant allocates 0 units): "
      f"{undefined_relative:,}")
print()
display(sensitivity_by_state)
print("Largest deviations by municipality:")
display(sensitivity_top_deviations.head(10))
display(sensitivity_top_deviations.tail(10))

Sensitivity variant: one- and two-dwelling residential buildings only
National sum (sensitivity)   : 407,637
Federal-state sums identical : True
Municipalities changed       : 2,019 of 2,092 (96.5%)
Total units redistributed    : 14,274 (3.50% of the national stock)
Largest positive deviation   : +50
Largest negative deviation   : -2,059
Relative difference undefined (main variant allocates 0 units): 0



,federal_state_id,federal_state_code,municipalities,state_stock_main,state_stock_sensitivity,municipalities_changed,share_of_municipalities_changed,sum_absolute_difference,mean_absolute_difference,median_absolute_difference,max_positive_difference,max_negative_difference,mean_absolute_relative_difference,redistributed_share_of_state_stock
0,1,Burgenland,171.0,26152.0,26152.0,150.0,0.877193,568.0,3.321637,2.0,9.0,-79.0,0.019149,0.010860
1,2,Kärnten,132.0,27636.0,27636.0,129.0,0.977273,2040.0,15.454545,7.0,40.0,-485.0,0.061210,0.036908
2,3,Niederösterreich,573.0,114759.0,114759.0,546.0,0.952880,5012.0,8.746946,5.0,50.0,-239.0,0.039633,0.021837
3,4,Oberösterreich,438.0,92713.0,92713.0,433.0,0.988584,7882.0,17.995434,9.0,41.0,-2059.0,0.067823,0.042508
4,5,Salzburg,119.0,21585.0,21585.0,115.0,0.966387,2508.0,21.075630,11.0,44.0,-881.0,0.102077,0.058096
5,6,Steiermark,285.0,50515.0,50515.0,282.0,0.989474,4540.0,15.929825,8.0,38.0,-1458.0,0.070980,0.044937
6,7,Tirol,277.0,32272.0,32272.0,269.0,0.971119,4144.0,14.960289,8.0,38.0,-1017.0,0.118513,0.064204
7,8,Vorarlberg,96.0,25661.0,25661.0,95.0,0.989583,1854.0,19.312500,9.0,45.0,-303.0,0.084147,0.036125
8,9,Wien,1.0,16344.0,16344.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000


Largest deviations by municipality:


,deviation_direction,municipality_id,municipality_name,federal_state_code,residential_buildings,buildings_1_2_dwellings,buildings_3_plus_dwellings,unit_count_main,unit_count_sensitivity,absolute_difference,relative_difference
0,largest_positive,31235,Gerasdorf bei Wien,Niederösterreich,5178.536,5070.768,107.768,1012,1062,50,0.049407
1,largest_positive,80410,Koblach,Vorarlberg,1535.840,1426.610,109.230,408,453,45,0.110294
2,largest_positive,50201,Abtenau,Salzburg,1613.200,1511.450,101.750,287,331,44,0.153310
3,largest_positive,80217,Höchst,Vorarlberg,2279.760,2037.616,242.144,605,648,43,0.071074
4,largest_positive,50337,Thalgau,Salzburg,1788.237,1649.754,138.483,319,361,42,0.131661
5,largest_positive,30856,Strasshof an der Nordbahn,Niederösterreich,4791.490,4664.865,126.625,936,977,41,0.043803
6,largest_positive,40702,Bad Goisern am Hallstättersee,Oberösterreich,2682.000,2524.060,157.940,645,686,41,0.063566
7,largest_positive,20913,St. Andrä,Kärnten,3205.809,3068.568,137.241,532,572,40,0.075188
8,largest_positive,80303,Lustenau,Vorarlberg,5259.969,4517.172,742.797,1396,1436,40,0.028653
9,largest_positive,50322,Lamprechtshausen,Salzburg,1213.700,1160.700,53.000,216,254,38,0.175926


,deviation_direction,municipality_id,municipality_name,federal_state_code,residential_buildings,buildings_1_2_dwellings,buildings_3_plus_dwellings,unit_count_main,unit_count_sensitivity,absolute_difference,relative_difference
30,largest_negative,30201,St. Pölten,Niederösterreich,11792.895,9920.790,1872.105,2305,2078,-227,-0.098482
31,largest_negative,30604,Baden,Niederösterreich,5345.120,4142.468,1202.652,1045,868,-177,-0.169378
32,largest_negative,30101,Krems an der Donau,Niederösterreich,5084.159,3918.125,1166.034,994,821,-173,-0.174044
33,largest_negative,31717,Mödling,Niederösterreich,3532.556,2477.220,1055.336,690,519,-171,-0.247826
34,largest_negative,61108,Leoben,Steiermark,3712.262,2350.950,1361.312,543,388,-155,-0.285451
35,largest_negative,80404,Feldkirch,Vorarlberg,7323.330,5668.890,1654.440,1944,1802,-142,-0.073045
36,largest_negative,70513,Kufstein,Tirol,2023.770,1038.282,985.488,388,253,-135,-0.347938
37,largest_negative,80103,Bludenz,Vorarlberg,2563.680,1737.540,826.140,680,552,-128,-0.188235
38,largest_negative,70354,Hall in Tirol,Tirol,1621.810,780.948,840.862,311,190,-121,-0.389068
39,largest_negative,40705,Gmunden,Oberösterreich,2782.188,2035.584,746.604,669,553,-116,-0.173393


## 13. Validation

All mandatory checks of the workflow are collected in a single table. Critical
checks gate the export in Section 14: no output file is written while any of
them fails.

In [15]:
validation_rows = []


def add_check(name, passed, expected=None, actual=None, critical=True):
    validation_rows.append({
        "check": name,
        "expected": expected,
        "actual": actual,
        "passed": bool(passed),
        "critical": bool(critical),
    })


# --- national control total ------------------------------------------------
add_check("national_stock_is_exactly_407637",
          NATIONAL_HEATING_HP_STOCK == 407_637, 407637, NATIONAL_HEATING_HP_STOCK)

# --- federal-state level ---------------------------------------------------
add_check("all_nine_federal_states_present",
          check_all_nine_states_present, 9, len(hp_households))
add_check("no_duplicate_federal_state_codes",
          check_no_duplicate_state_codes, True, check_no_duplicate_state_codes)
add_check("no_negative_hp_household_values",
          check_no_negative_households, True, check_no_negative_households)
add_check("state_household_sum_matches_national_sheet",
          check_state_sum_matches_national_sheet, 0.0,
          round(household_sum_deviation, 9))
add_check("state_weights_sum_to_one",
          check_state_shares_sum_to_one, 1.0,
          float(hp_households["hp_household_share"].sum()))
add_check("state_stock_sums_to_national",
          check_state_stock_sums_to_national, NATIONAL_HEATING_HP_STOCK,
          int(federal_state_audit["hp_stock_final"].sum()))

# --- energy validation (reported, never gating the allocation) -------------
add_check("energy_internal_consistency_E_equals_H_times_e",
          check_energy_internal_consistency, True,
          check_energy_internal_consistency, critical=False)
add_check("energy_cross_source_consistency",
          check_energy_cross_source_consistency, True,
          check_energy_cross_source_consistency, critical=False)
add_check("energy_cross_source_deviation_explained_by_rounding",
          check_cross_source_rounding_explained, True,
          check_cross_source_rounding_explained, critical=False)

# --- municipality master ---------------------------------------------------
add_check("municipality_id_unique_in_master",
          check_municipality_id_unique, True, check_municipality_id_unique)
add_check("municipality_count_2092",
          check_municipality_count, 2092, len(municipalities))
add_check("vienna_present_exactly_once",
          check_vienna_present_once, True, check_vienna_present_once)
add_check("no_missing_federal_state_assignment",
          check_state_assignment_complete, True, check_state_assignment_complete)

# --- building proxy --------------------------------------------------------
add_check("no_negative_building_counts",
          check_no_negative_buildings, True, check_no_negative_buildings)
add_check("building_class_shares_sum_to_100",
          check_building_shares_sum_to_100, "<= 0.35 pp",
          round(max_share_sum_deviation, 3), critical=False)
add_check("all_municipalities_have_building_data",
          check_all_municipalities_have_building_data, len(municipalities),
          len(building_proxy))
add_check("no_unknown_municipalities_in_building_source",
          check_no_unknown_aest_municipalities, 0,
          len(unknown_aest_municipalities))
add_check("no_unmatched_records",
          check_no_unmatched_records, 0, len(unmatched_records))
add_check("building_proxy_reconstructs_state_rows",
          check_building_proxy_reconstructs_state_rows,
          f"<= {BUILDING_STATE_TOLERANCE:.0%}",
          f"{max_building_state_deviation:.4%}")
add_check("no_federal_state_with_zero_proxy_total",
          check_no_zero_state_proxy_total, True, check_no_zero_state_proxy_total)

# --- municipal allocation --------------------------------------------------
add_check("no_rows_dropped_during_allocation",
          check_row_count_preserved, len(municipalities), len(municipality_result))
add_check("all_canonical_municipalities_present_and_no_extras",
          check_municipality_ids_unchanged, True, check_municipality_ids_unchanged)
add_check("municipality_id_unique_in_output",
          check_municipality_id_unique_in_output, True,
          check_municipality_id_unique_in_output)
add_check("unit_count_is_integer",
          check_unit_count_integer, True, check_unit_count_integer)
add_check("unit_count_non_negative",
          check_unit_count_non_negative, True, check_unit_count_non_negative)
add_check("unit_count_not_missing",
          check_unit_count_not_missing, True, check_unit_count_not_missing)

for state_id in sorted(STATE_STOCK):
    state_name = GKZ_PREFIX_TO_STATE_NAME[state_id]
    add_check(
        f"municipal_sum_matches_state_stock_{state_name}",
        int(state_sum_check[state_id]) == int(STATE_STOCK[state_id]),
        int(STATE_STOCK[state_id]),
        int(state_sum_check[state_id]),
    )

add_check("national_municipal_sum_is_407637",
          check_national_municipal_sum, NATIONAL_HEATING_HP_STOCK,
          national_municipal_sum)

# --- sensitivity variant ---------------------------------------------------
add_check("sensitivity_state_sums_preserved",
          check_sensitivity_state_sums_preserved, True,
          check_sensitivity_state_sums_preserved)
add_check("sensitivity_national_sum_is_407637",
          check_sensitivity_national_sum, NATIONAL_HEATING_HP_STOCK,
          int(sensitivity_result["unit_count"].sum()))

# --- no silent data handling ----------------------------------------------
add_check("no_silently_dropped_source_rows",
          len(aest_row_usage) > 0 and bool(aest_row_usage["usage"].notna().all()),
          True, True)
add_check("no_silent_imputation_of_building_values",
          bool(municipality_building_proxy["residential_buildings"].notna().all()),
          True, True)
add_check("no_silent_name_correction",
          bool((municipality_result["municipality_name"].values
                == municipalities.sort_values("municipality_id")
                ["municipality_name"].values).all()),
          True, True)

validation_report = pd.DataFrame(validation_rows)
display(validation_report)

critical_checks = validation_report.loc[validation_report["critical"]]
all_critical_checks_passed = bool(critical_checks["passed"].all())
all_checks_passed = bool(validation_report["passed"].all())

print()
print(f"Checks run: {len(validation_report)} "
      f"({int(validation_report['critical'].sum())} critical, "
      f"{int((~validation_report['critical']).sum())} reported)")
print("ALL CRITICAL CHECKS PASSED:", all_critical_checks_passed)
print("ALL CHECKS PASSED:", all_checks_passed)
if not all_checks_passed:
    print("Failed checks:",
          validation_report.loc[~validation_report["passed"], "check"].tolist())

,check,expected,actual,passed,critical
0,national_stock_is_exactly_407637,407637,407637,True,True
1,all_nine_federal_states_present,9,9,True,True
2,no_duplicate_federal_state_codes,True,True,True,True
3,no_negative_hp_household_values,True,True,True,True
4,state_household_sum_matches_national_sheet,0.0,-0.0,True,True
5,state_weights_sum_to_one,1.0,1.0,True,True
6,state_stock_sums_to_national,407637,407637,True,True
7,energy_internal_consistency_E_equals_H_times_e,True,True,True,False
8,energy_cross_source_consistency,True,True,True,False
9,energy_cross_source_deviation_explained_by_rou...,True,True,True,False



Checks run: 42 (38 critical, 4 reported)
ALL CRITICAL CHECKS PASSED: True
ALL CHECKS PASSED: True


## 14. Export harmonised outputs (blocked while critical checks fail)

Output formats follow the sibling PV and EV workflows: one multi-sheet Excel
workbook with the canonical municipality table plus all audit and diagnostic
tables, and one Parquet file holding the complete municipality-level audit
trail.

Canonical schema of sheet `hp_by_municipality`:

```text
municipality_id, municipality_name, federal_state_code, reference_date,
unit_count, derivation_method, data_quality
```

`unit_count` is the final integer municipal heat-pump stock $N_m^{HP}$;
`reference_date` is 2024-12-31; the federal-state proxy originates from the
statistical survey period 2023/2024, which is recorded separately in the audit
tables and in `source_metadata` and is deliberately not equated with the
reference date.

In [ ]:
if all_critical_checks_passed:
    canonical_columns = [
        "municipality_id",
        "municipality_name",
        "federal_state_code",
        "reference_date",
        "unit_count",
        "derivation_method",
        "data_quality",
    ]
    final_output = (
        municipality_result[canonical_columns]
        .sort_values("municipality_id")
        .reset_index(drop=True)
    )

    municipality_audit = municipality_result[
        [
            "municipality_id", "municipality_name", "federal_state_id",
            "federal_state_code", "state_hp_stock_final",
            "buildings_total_all_types", "buildings_1_2_dwellings",
            "buildings_3_plus_dwellings", "residential_buildings",
            "municipal_proxy_value", "state_proxy_total",
            "proxy_share_within_state", "hp_stock_raw", "hp_stock_floor",
            "fractional_remainder", "rounding_adjustment", "unit_count",
            "proxy_variant", "building_proxy_resolution_rule",
            "aest_source_code", "aest_source_label", "reference_date",
            "derivation_method", "data_quality",
        ]
    ].merge(
        sensitivity_comparison[
            ["municipality_id", "sensitivity_proxy_value",
             "sensitivity_proxy_share_within_state", "sensitivity_hp_stock_raw",
             "unit_count_sensitivity", "absolute_difference", "relative_difference"]
        ].rename(columns={
            "absolute_difference": "sensitivity_absolute_difference",
            "relative_difference": "sensitivity_relative_difference",
        }),
        on="municipality_id",
        how="left",
        validate="one_to_one",
    )
    municipality_audit["survey_period_state_proxy"] = STATE_PROXY_SURVEY_PERIOD
    municipality_audit["building_proxy_year"] = BUILDING_PROXY_YEAR
    municipality_audit["value_status"] = "modelled_estimate_not_observed_stock"

    source_metadata = pd.DataFrame(
        [
            {
                "source_file": MARKET_REPORT_PATH_FOR_CITATION_ONLY.name,
                "publisher": "Bundesministerium für Innovation, Mobilität und Infrastruktur (BMIMI)",
                "reference_period": "2024",
                "spatial_resolution": "Austria (national)",
                "sheet_or_table": "Chapter 9 Wärmepumpen, Section 9.1.3, Table 51, row 2024, column 'Heizung'",
                "role": "National heating-heat-pump stock (407,637), transcribed manually into the configuration cell; the file is NOT parsed by the notebook",
                "limitations": "Modelled stock derived from historical sales and an assumed 20-year technical service life; no Weibull retirement function; not an installation census",
            },
            {
                "source_file": HP_HOUSEHOLDS_PATH.name,
                "publisher": "STATISTIK AUSTRIA, Energiestatistik (MZ Energieeinsatz der Haushalte)",
                "reference_period": STATE_PROXY_SURVEY_PERIOD,
                "spatial_resolution": "federal state",
                "sheet_or_table": "one sheet per federal state, block 'Gesamteinsatz aller Energieträger 2023/2024', row 'Wärmepumpe', columns 'Haushalte (HH)', 'GJ', 'GJ pro HH'",
                "role": "Stage 1 allocation proxy: reported number of households using a heat pump per federal state; GJ columns are validation only",
                "limitations": "Households, not installations; households are counted once per energy carrier used, so carriers overlap; survey period 2023/2024 differs from the 2024 stock reference date",
            },
            {
                "source_file": HP_ENERGY_VALIDATION_PATH.name,
                "publisher": "STATISTIK AUSTRIA, STATcube",
                "reference_period": STATE_PROXY_SURVEY_PERIOD,
                "spatial_resolution": "federal state",
                "sheet_or_table": "'Datenblatt 0', row 'Wärmepumpe', columns Burgenland..Wien",
                "role": "Validation only: cross-source comparison of federal-state heat-pump energy input; never affects the allocation",
                "limitations": "Published as whole GJ, so small rounding deviations against the full-precision household source are expected",
            },
            {
                "source_file": BUILDINGS_PATH.name,
                "publisher": "STATISTIK AUSTRIA, Abgestimmte Erwerbsstatistik 2024",
                "reference_period": str(BUILDING_PROXY_YEAR),
                "spatial_resolution": f"municipality (territorial status {BUILDING_PROXY_TERRITORIAL_STATUS})",
                "sheet_or_table": f"'{BUILDINGS_SHEET}', block 'Gebäudebestand': 'Gebäude gesamt', 'Gebäude mit 1 oder 2 Wohnungen in %', 'Gebäude mit 3 oder mehr Wohnungen in %', 'Andere Gebäude in %'",
                "role": "Stage 2 allocation proxy: residential buildings per municipality (main variant) and one-/two-dwelling buildings (sensitivity variant)",
                "limitations": "Building classes are published as percentage shares rounded to one decimal place, so class counts are derived and carry a rounding uncertainty; 14 statutory cities and Vienna are resolved from documented aggregate rows",
            },
            {
                "source_file": MUNICIPALITIES_PATH.name,
                "publisher": "STATISTIK AUSTRIA (via the sibling EV harmonisation workflow)",
                "reference_period": "territorial status 2026",
                "spatial_resolution": "municipality",
                "sheet_or_table": "'municipalities'",
                "role": "Canonical municipality master: Gemeindekennziffer, municipality name, federal-state code; the only join key of this workflow",
                "limitations": "Unchanged byte-identical copy of harmonisation_EV/data/municipalities_clean.xlsx; the EV file itself is never read or modified here",
            },
        ]
    )

    try:
        with pd.ExcelWriter(OUTPUT_XLSX_PATH, engine="openpyxl") as writer:
            final_output.to_excel(writer, sheet_name="hp_by_municipality", index=False)
            federal_state_audit.to_excel(
                writer, sheet_name="federal_state_audit", index=False
            )
            validation_report.to_excel(writer, sheet_name="validation", index=False)
            energy_cross_source_validation.to_excel(
                writer, sheet_name="energy_cross_source_validation", index=False
            )
            building_proxy_rule_summary.to_excel(
                writer, sheet_name="building_proxy_rule_summary", index=False
            )
            building_proxy_special_rows.to_excel(
                writer, sheet_name="building_proxy_special_rows", index=False
            )
            building_proxy_state_validation.to_excel(
                writer, sheet_name="building_proxy_state_valid", index=False
            )
            municipality_name_comparison.to_excel(
                writer, sheet_name="municipality_name_comparison", index=False
            )
            unmatched_records.to_excel(
                writer, sheet_name="unmatched_records", index=False
            )
            aest_row_usage.to_excel(writer, sheet_name="aest_row_usage", index=False)
            sensitivity_by_state.to_excel(
                writer, sheet_name="sensitivity_by_state", index=False
            )
            sensitivity_top_deviations.to_excel(
                writer, sheet_name="sensitivity_top_deviations", index=False
            )
            source_metadata.to_excel(writer, sheet_name="source_metadata", index=False)

            for cell in writer.sheets["hp_by_municipality"]["A"][1:]:
                cell.number_format = "@"

    except PermissionError as error:
        raise PermissionError(
            f"Could not write {OUTPUT_XLSX_PATH}. Close the file in Excel (or any "
            "other program) and rerun this cell."
        ) from error

    municipality_audit.to_parquet(AUDIT_PARQUET_PATH, index=False)

    print("Created:", OUTPUT_XLSX_PATH.resolve())
    print("Created:", AUDIT_PARQUET_PATH.resolve())
    print()
    print("Canonical output preview:")
    display(final_output.head())
else:
    print("FINAL EXPORT BLOCKED.")
    print("Reason: at least one critical validation check failed.")
    print("Failed critical checks:",
          critical_checks.loc[~critical_checks["passed"], "check"].tolist())

Created: C:\Users\friha\TU Wien\Diplomarbeit\00_Harmonisierung\00_Harmonisierung_HP\outputs\hp_municipality_harmonised.xlsx
Created: C:\Users\friha\TU Wien\Diplomarbeit\00_Harmonisierung\00_Harmonisierung_HP\outputs\hp_municipality_audit.parquet

Canonical output preview:


,municipality_id,municipality_name,federal_state_code,reference_date,unit_count,derivation_method,data_quality
0,10101,Eisenstadt,Burgenland,2024-12-31,766,national_heating_hp_stock_state_hp_households_...,modelled_mass_preserving
1,10201,Rust,Burgenland,2024-12-31,253,national_heating_hp_stock_state_hp_households_...,modelled_mass_preserving
2,10301,Breitenbrunn am Neusiedler See,Burgenland,2024-12-31,214,national_heating_hp_stock_state_hp_households_...,modelled_mass_preserving
3,10302,Donnerskirchen,Burgenland,2024-12-31,189,national_heating_hp_stock_state_hp_households_...,modelled_mass_preserving
4,10303,Großhöflein,Burgenland,2024-12-31,174,national_heating_hp_stock_state_hp_households_...,modelled_mass_preserving


## 15. Result and QA summary

In [17]:
print("=" * 72)
print("HP STOCK HARMONISATION - RESULT AND QA SUMMARY")
print("=" * 72)
print()
print("Target quantity : heating heat pumps in operation per Austrian municipality")
print(f"Reference year  : {REFERENCE_YEAR}")
print(f"Reference date  : {REFERENCE_DATE}")
print(f"Excluded        : domestic-hot-water-only, ventilation and industrial heat pumps")
print()
print("--- Inputs -----------------------------------------------------------")
print(f"National stock (manually configured) : {NATIONAL_HEATING_HP_STOCK:,}")
print(f"Federal-state proxy                  : heat-pump households, "
      f"survey period {STATE_PROXY_SURVEY_PERIOD}")
print(f"Municipal proxy (main)               : residential buildings, "
      f"AEST {BUILDING_PROXY_YEAR}, territorial status {BUILDING_PROXY_TERRITORIAL_STATUS}")
print(f"Municipal proxy (sensitivity)        : one- and two-dwelling buildings only")
print()
print("--- Stage 1: Austria -> federal states -------------------------------")
for row in federal_state_audit.itertuples(index=False):
    print(f"  {row.federal_state_id}  {row.federal_state_code:<18s} "
          f"households {row.hp_households:>12,.1f}  "
          f"share {row.hp_household_share:>7.4f}  "
          f"stock {row.hp_stock_final:>8,}")
print(f"  {'':>3s}{'TOTAL':<19s} "
      f"households {federal_state_audit['hp_households'].sum():>12,.1f}  "
      f"share {federal_state_audit['hp_household_share'].sum():>7.4f}  "
      f"stock {int(federal_state_audit['hp_stock_final'].sum()):>8,}")
print()
print("--- Stage 2: federal states -> municipalities ------------------------")
print(f"  Municipalities in output            : {len(municipality_result):,}")
print(f"  National sum of unit_count          : {national_municipal_sum:,}")
print(f"  Municipalities with unit_count = 0  : "
      f"{int((municipality_result['unit_count'] == 0).sum()):,}")
print(f"  Median unit_count                   : "
      f"{municipality_result['unit_count'].median():,.0f}")
print(f"  Maximum unit_count                  : "
      f"{int(municipality_result['unit_count'].max()):,} "
      f"({municipality_result.loc[municipality_result['unit_count'].idxmax(), 'municipality_name']})")
print()
print("--- Mass preservation ------------------------------------------------")
print(f"  Sum of federal-state stocks = {NATIONAL_HEATING_HP_STOCK:,} : "
      f"{check_state_stock_sums_to_national}")
print(f"  Every federal state's municipal sum = its stock        : "
      f"{check_state_sums_match}")
print(f"  National municipal sum = {NATIONAL_HEATING_HP_STOCK:,}            : "
      f"{check_national_municipal_sum}")
print(f"  Sensitivity variant preserves the same totals          : "
      f"{check_sensitivity_state_sums_preserved and check_sensitivity_national_sum}")
print()
print("--- Sensitivity ------------------------------------------------------")
print(f"  Municipalities with a different value : "
      f"{int((sensitivity_comparison['absolute_difference'] != 0).sum()):,} "
      f"({(sensitivity_comparison['absolute_difference'] != 0).mean():.1%})")
print(f"  Units redistributed                   : "
      f"{int(sensitivity_comparison['absolute_difference'].abs().sum()) // 2:,} "
      f"({sensitivity_comparison['absolute_difference'].abs().sum() / (2 * NATIONAL_HEATING_HP_STOCK):.2%} "
      "of the national stock)")
print(f"  Deviation range                       : "
      f"{int(sensitivity_comparison['absolute_difference'].min()):+,} to "
      f"{int(sensitivity_comparison['absolute_difference'].max()):+,} units")
print()
print("--- Diagnostics ------------------------------------------------------")
print(f"  Unmatched / unresolvable records      : {len(unmatched_records)}")
print(f"  Unknown municipalities in the source  : {len(unknown_aest_municipalities)}")
print(f"  Differing source labels (reported)    : {len(municipality_name_comparison)}")
print(f"  Source rows recorded in aest_row_usage: {len(aest_row_usage)}")
print()
print("--- Quality assurance ------------------------------------------------")
print(f"  Checks run                            : {len(validation_report)}")
print(f"  Critical checks passed                : {all_critical_checks_passed}")
print(f"  All checks passed                     : {all_checks_passed}")
print(f"  Final export completed                : {all_critical_checks_passed}")
print()
print("All federal-state and municipality values are MODELLED ESTIMATES.")
print("They are not observed heat-pump counts and do not reconstruct the")
print("actual installation sites.")
print("=" * 72)

HP STOCK HARMONISATION - RESULT AND QA SUMMARY

Target quantity : heating heat pumps in operation per Austrian municipality
Reference year  : 2024
Reference date  : 2024-12-31
Excluded        : domestic-hot-water-only, ventilation and industrial heat pumps

--- Inputs -----------------------------------------------------------
National stock (manually configured) : 407,637
Federal-state proxy                  : heat-pump households, survey period 2023/2024
Municipal proxy (main)               : residential buildings, AEST 2024, territorial status 2026
Municipal proxy (sensitivity)        : one- and two-dwelling buildings only

--- Stage 1: Austria -> federal states -------------------------------
  1  Burgenland         households     39,639.1  share  0.0642  stock   26,152
  2  Kärnten            households     41,888.7  share  0.0678  stock   27,636
  3  Niederösterreich   households    173,941.8  share  0.2815  stock  114,759
  4  Oberösterreich     households    140,525.8  share  0